# Notebook 08 v3 — DINOv2 + ViT-B/16 Direct Inference Ensemble

**Submission notebook** cho RSNA Knee Abnormality Detection. Notebook tự đọc test DICOM, chạy inference từ hai model artifact đã train, kết hợp xác suất và tạo `/kaggle/working/submission.csv` khi chạy.

## Inputs cần attach trong Kaggle

1. Competition: **RSNA Knee Abnormality Detection**.
2. Dataset DINOv2: `five-role-v2-cv5-submission-bundle-zip` — dạng thư mục đã giải nén, có `bundle_manifest.json`, `dinov2_repo/` và `five_role_2p5d_v2_cv5/`.
3. Dataset ViT: `vit-pretrained-centerprior-model-artifacts-zip` — dạng thư mục đã giải nén, có `vit_pretrained_best_checkpoint.pt`.

Notebook này đã cấu hình sẵn đường dẫn theo hai Dataset trên. Không cần ZIP, không cần submission CSV cũ, không cần train lại model. Nếu tên Dataset/đường dẫn mount trên Kaggle của bạn khác, chỉ sửa các biến `DINO_BUNDLE_OVERRIDE` và `VIT_ARTIFACT_OVERRIDE` ở Cell 1.

**Runtime:** bật GPU, Internet OFF. Notebook chạy inference trên test DICOM và tạo submission; không thực hiện training.

In [19]:
# ============================================================
# CELL 1 — Shared imports, settings and artifact overrides
# ============================================================
import os, re, gc, json, math, time, copy, shutil, zipfile, warnings
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import pydicom
from pydicom.pixels import apply_modality_lut

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vit_b_16
from IPython.display import display, FileLink

warnings.filterwarnings("ignore", category=UserWarning)

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
OUTPUT_DIR = WORK_ROOT / "model_ensemble_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ID_COL = "StudyInstanceUID"
TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]
EXPECTED_COLUMNS = [ID_COL, *TARGETS]

# Paths below match the two extracted Kaggle Datasets attached to this notebook.
# You may leave them unchanged when using the same Dataset slugs.
DINO_BUNDLE_OVERRIDE = "/kaggle/input/datasets/biinseventeen/five-role-v2-cv5-submission-bundle-zip"  # extracted Dataset directory
VIT_ARTIFACT_OVERRIDE = "/kaggle/input/datasets/biinseventeen/vit-pretrained-centerprior-model-artifacts-zip/vit_pretrained_best_checkpoint.pt"  # direct checkpoint file
COMP_ROOT_OVERRIDE = ""    # exact directory containing test.csv

# Main submission blend. Candidates are also written for later leaderboard comparison.
DINO_WEIGHT = 0.90
CANDIDATE_DINO_WEIGHTS = [1.00, 0.90, 0.80, 0.70]

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("GPU is required. Enable GPU in Kaggle Settings → Accelerator.")
device = torch.device("cuda")
USE_AMP = True
print("PyTorch:", torch.__version__)
print("OpenCV:", cv2.__version__)
print("pydicom:", pydicom.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("DINO blend weight:", DINO_WEIGHT, "| ViT weight:", 1.0-DINO_WEIGHT)
print("Internet must be OFF for the committed submission run.")

PyTorch: 2.10.0+cu128
OpenCV: 4.13.0
pydicom: 3.0.2
GPU: Tesla T4
DINO blend weight: 0.9 | ViT weight: 0.09999999999999998
Internet must be OFF for the committed submission run.


## 2. Competition test data and submission schema

In [20]:
# Locate the exact competition input; no prediction CSV is read here.
if COMP_ROOT_OVERRIDE:
    COMP_ROOT = Path(COMP_ROOT_OVERRIDE)
else:
    candidate_roots = []
    comp_base = INPUT_ROOT / "competitions"
    if comp_base.is_dir():
        candidate_roots = [p for p in comp_base.iterdir() if (p / "test.csv").is_file()]
    candidate_roots = candidate_roots or [p.parent for p in INPUT_ROOT.rglob("test.csv") if (p.parent / "sample_submission.csv").is_file()]
    candidate_roots = list({str(p.resolve()): p for p in candidate_roots}.values())
    if len(candidate_roots) != 1:
        raise FileNotFoundError(f"Could not uniquely locate competition test data: {candidate_roots}")
    COMP_ROOT = candidate_roots[0]

TEST_CSV = COMP_ROOT / "test.csv"
TEST_SERIES_CSV = COMP_ROOT / "test_series.csv"
SAMPLE_SUBMISSION_CSV = COMP_ROOT / "sample_submission.csv"
TEST_SERIES_DIR = next((p for p in [COMP_ROOT/"test_series", COMP_ROOT/"test_images", COMP_ROOT/"test_dicom"] if p.is_dir()), None)
for p in [TEST_CSV, TEST_SERIES_CSV, SAMPLE_SUBMISSION_CSV]:
    if not p.is_file():
        raise FileNotFoundError(p)
if TEST_SERIES_DIR is None:
    raise FileNotFoundError("Cannot find test_series/ DICOM directory.")

test_df = pd.read_csv(TEST_CSV, dtype={ID_COL: str})
test_series_df = pd.read_csv(TEST_SERIES_CSV, dtype={ID_COL: str, "SeriesInstanceUID": str})
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_CSV, dtype={ID_COL: str})
for df in [test_df, test_series_df, sample_submission]:
    df[ID_COL] = df[ID_COL].astype(str)
if sample_submission.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(f"sample_submission schema/order mismatch: {sample_submission.columns.tolist()}")
if test_df[ID_COL].duplicated().any() or sample_submission[ID_COL].duplicated().any():
    raise ValueError("Duplicate StudyInstanceUID in test.csv or sample_submission.csv.")
if set(test_df[ID_COL]) != set(sample_submission[ID_COL]):
    raise ValueError("test.csv and sample_submission.csv UID sets differ.")
required_series_cols = {ID_COL, "SeriesInstanceUID", "Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"}
missing_series_cols = required_series_cols - set(test_series_df.columns)
if missing_series_cols:
    raise ValueError(f"test_series.csv missing columns: {sorted(missing_series_cols)}")

test_uids = test_df[ID_COL].astype(str).tolist()
N_TEST = len(test_uids)
print("Competition:", COMP_ROOT)
print("Test DICOM:", TEST_SERIES_DIR)
print("Test studies:", N_TEST, "| series rows:", len(test_series_df))
print("Submission schema validated:", EXPECTED_COLUMNS)
if N_TEST <= 3:
    print("Note: Kaggle may replace this visible sample test set with hidden scoring studies during commit.")

Competition: /kaggle/input/competitions/rsna-knee-abnormality-detection
Test DICOM: /kaggle/input/competitions/rsna-knee-abnormality-detection/test_series
Test studies: 3 | series rows: 15
Submission schema validated: ['StudyInstanceUID', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
Note: Kaggle may replace this visible sample test set with hidden scoring studies during commit.


## 3. Locate ViT-B/16 checkpoint

Cell này ưu tiên đọc trực tiếp `vit_pretrained_best_checkpoint.pt` từ Dataset đã attach. ZIP vẫn được hỗ trợ nếu Dataset của bạn cung cấp ZIP. Nếu đường dẫn mount thay đổi giữa phiên chạy thường và phiên submit, notebook tự tìm file trong `/kaggle/input` theo tên chính xác. Checkpoint không bị copy sang `/kaggle/working`.

In [21]:
VIT_ZIP_NAME = "vit_pretrained_centerprior_model_artifacts.zip"
VIT_CKPT_NAME = "vit_pretrained_best_checkpoint.pt"


def find_named_files(roots, filename):
    found = {}
    skip = {
        "train_series", "test_series", "train_images", "test_images",
        "preprocessed_cache", "test_preprocessed_cache", ".git", "__pycache__",
    }
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for current, dirs, files in os.walk(root):
            dirs[:] = [d for d in dirs if d.lower() not in skip]
            if filename in files:
                path = Path(current) / filename
                found[str(path.resolve())] = path
    return list(found.values())


def resolve_artifact(override, filename, label):
    if override:
        path = Path(override)
        if path.is_file():
            return path
        if path.is_dir():
            candidates = find_named_files([path], filename)
            if len(candidates) == 1:
                return candidates[0]

    # Discover in attached Kaggle Datasets first. Only then use File Persistence.
    candidates = find_named_files([INPUT_ROOT], filename)
    if not candidates:
        candidates = find_named_files([WORK_ROOT], filename)
    if len(candidates) != 1:
        detail = "\n".join(str(path) for path in candidates) or "(none)"
        override_note = f"\nConfigured override: {override}" if override else ""
        raise RuntimeError(
            f"Expected exactly one {label}; found {len(candidates)}:\n{detail}"
            f"{override_note}\nCheck that the correct Kaggle Dataset is attached."
        )
    return candidates[0]


# The Dataset currently contains a standalone .pt, not a ZIP archive.
# If Kaggle mounts it at a different path in the committed run, fall back to
# searching the attached /kaggle/input tree by exact checkpoint filename.
vit_artifact_path = None
if VIT_ARTIFACT_OVERRIDE:
    candidate = Path(VIT_ARTIFACT_OVERRIDE)
    if candidate.is_file():
        vit_artifact_path = candidate

if vit_artifact_path is None:
    vit_candidates = find_named_files([INPUT_ROOT], VIT_CKPT_NAME)
    if not vit_candidates:
        vit_candidates = find_named_files([INPUT_ROOT], VIT_ZIP_NAME)
    if not vit_candidates:
        vit_candidates = (
            find_named_files([WORK_ROOT], VIT_CKPT_NAME)
            or find_named_files([WORK_ROOT], VIT_ZIP_NAME)
        )
    vit_candidates = list({str(p.resolve()): p for p in vit_candidates}.values())
    if len(vit_candidates) != 1:
        details = "\n".join(str(p) for p in vit_candidates) or "(none)"
        raise RuntimeError(
            "Expected exactly one ViT best-checkpoint file or artifact ZIP; "
            f"found {len(vit_candidates)}:\n{details}\n"
            f"Configured override: {VIT_ARTIFACT_OVERRIDE or '(empty)'}\n"
            "Verify that the ViT Dataset is attached to this notebook version."
        )
    vit_artifact_path = vit_candidates[0]

if vit_artifact_path.suffix.lower() == ".zip":
    # Keep weights inside the read-only Input ZIP; no checkpoint copy in working.
    VIT_ZIP_HANDLE = zipfile.ZipFile(vit_artifact_path, "r")
    vit_members = [
        member for member in VIT_ZIP_HANDLE.namelist()
        if Path(member).name == VIT_CKPT_NAME
    ]
    if len(vit_members) != 1:
        VIT_ZIP_HANDLE.close()
        raise FileNotFoundError(
            f"Expected exactly one {VIT_CKPT_NAME} in {vit_artifact_path}; found {vit_members}"
        )
    VIT_CKPT_MEMBER = vit_members[0]
    BEST_CKPT = VIT_ZIP_HANDLE.open(VIT_CKPT_MEMBER, "r")
else:
    if vit_artifact_path.name != VIT_CKPT_NAME:
        raise FileNotFoundError(
            f"Resolved ViT file is not {VIT_CKPT_NAME}: {vit_artifact_path}"
        )
    VIT_ZIP_HANDLE = None
    BEST_CKPT = vit_artifact_path

# Exact inference settings from the ViT-B/16 submission pipeline.
PLANE_ORDER = ["Sagittal", "Coronal", "Axial"]
PLANE_TO_ID = {"Sagittal": 0, "Coronal": 1, "Axial": 2}
IMG_SIZE = 224
PATCH_SIZE = 16
MAX_TOTAL_SLICES = 36
SLICES_PER_SERIES = 12
MAX_SERIES_PER_STUDY = 3
MIN_DECODED_SLICES_PER_SERIES = 4
SLICE_CHUNK_SIZE = 8
BATCH_SIZE = 1
NUM_WORKERS = 2
PREPROCESS_SAVE_EVERY = 10
USE_AMP = True
OUTPUT_ROOT = OUTPUT_DIR / "vit_inference"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUTPUT_ROOT / "test_preprocessed_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("ViT artifact:", vit_artifact_path)
print("ViT checkpoint:", VIT_CKPT_MEMBER if "VIT_CKPT_MEMBER" in globals() else BEST_CKPT)
print("ViT test cache:", CACHE_DIR)

ViT artifact: /kaggle/input/datasets/biinseventeen/vit-pretrained-centerprior-model-artifacts-zip/vit_pretrained_best_checkpoint.pt
ViT checkpoint: /kaggle/input/datasets/biinseventeen/vit-pretrained-centerprior-model-artifacts-zip/vit_pretrained_best_checkpoint.pt
ViT test cache: /kaggle/working/model_ensemble_v2/vit_inference/test_preprocessed_cache


## 4. ViT-B/16: select one series per plane

In [22]:
test_series_df["SeriesInstanceUID"] = test_series_df["SeriesInstanceUID"].astype(str)
test_series_df["plane_norm"] = test_series_df["Anatomical_Plane"].astype(str).str.strip().str.title()
test_series_df["fluid_num"] = pd.to_numeric(test_series_df["Fluid_Sensitive"], errors="coerce").fillna(-1)
test_series_df["fat_num"] = pd.to_numeric(test_series_df["Fat_Suppression"], errors="coerce").fillna(-1)

selected_rows = []
selected_by_uid = defaultdict(list)
plane_presence_rows = []

for uid, group in test_series_df.groupby(ID_COL, sort=False):
    present = []
    for plane in PLANE_ORDER:
        candidates = group[group["plane_norm"].eq(plane)].copy()
        if candidates.empty:
            continue
        candidates = candidates.sort_values(
            ["fluid_num", "fat_num", "SeriesInstanceUID"],
            ascending=[False, False, True],
            kind="mergesort",
        )
        chosen = candidates.iloc[0]
        item = {
            ID_COL: str(uid),
            "SeriesInstanceUID": str(chosen["SeriesInstanceUID"]),
            "Anatomical_Plane": plane,
            "plane_id": PLANE_TO_ID[plane],
            "Fluid_Sensitive": int(chosen["fluid_num"]) if chosen["fluid_num"] in (0, 1) else -1,
            "Fat_Suppression": int(chosen["fat_num"]) if chosen["fat_num"] in (0, 1) else -1,
        }
        selected_rows.append(item)
        selected_by_uid[str(uid)].append(item)
        present.append(plane)
    plane_presence_rows.append({ID_COL: str(uid), "n_planes_selected": len(present), "planes": ";".join(present)})

selected_test_series = pd.DataFrame(selected_rows)
plane_presence_df = pd.DataFrame(plane_presence_rows)
selected_test_series.to_csv(OUTPUT_ROOT / "selected_test_series.csv", index=False)

print("Selected series rows:", len(selected_test_series))
print("Studies with 3 planes:", int((plane_presence_df.n_planes_selected == 3).sum()))
print("Studies with 2 planes:", int((plane_presence_df.n_planes_selected == 2).sum()))
print("Studies with 1 plane:", int((plane_presence_df.n_planes_selected == 1).sum()))
print("Studies with 0 planes:", int((plane_presence_df.n_planes_selected == 0).sum()))
display(plane_presence_df.head())

Selected series rows: 9
Studies with 3 planes: 3
Studies with 2 planes: 0
Studies with 1 plane: 0
Studies with 0 planes: 0


,StudyInstanceUID,n_planes_selected,planes
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,3,Sagittal;Coronal;Axial
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,3,Sagittal;Coronal;Axial
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,3,Sagittal;Coronal;Axial


## 5. ViT-B/16: DICOM preprocessing functions

In [23]:
def _safe_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return default


def read_sorted_dicom_headers(series_dir: Path):
    paths = sorted([
        p for p in series_dir.iterdir()
        if p.is_file() and p.suffix.lower() == ".dcm"
    ])
    if not paths:
        return []

    rows = []
    first_normal = None
    for path in paths:
        try:
            ds = pydicom.dcmread(
                str(path), stop_before_pixels=True, force=True,
                specific_tags=["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]
            )
            iop = getattr(ds, "ImageOrientationPatient", None)
            ipp = getattr(ds, "ImagePositionPatient", None)
            instance = _safe_float(getattr(ds, "InstanceNumber", np.nan))

            position_value = np.nan
            if iop is not None and ipp is not None and len(iop) >= 6 and len(ipp) >= 3:
                row_cos = np.asarray(iop[:3], dtype=np.float64)
                col_cos = np.asarray(iop[3:6], dtype=np.float64)
                normal = np.cross(row_cos, col_cos)
                norm = np.linalg.norm(normal)
                if norm > 0:
                    normal = normal / norm
                    if first_normal is None:
                        first_normal = normal
                    position_value = float(np.dot(np.asarray(ipp[:3], dtype=np.float64), first_normal))

            rows.append({
                "path": path,
                "filename": path.name,
                "position": position_value,
                "instance": instance,
            })
        except Exception as exc:
            rows.append({
                "path": path, "filename": path.name,
                "position": np.nan, "instance": np.nan,
                "header_error": f"{type(exc).__name__}: {exc}",
            })

    valid_pos = sum(np.isfinite(r.get("position", np.nan)) for r in rows)
    valid_inst = sum(np.isfinite(r.get("instance", np.nan)) for r in rows)

    if valid_pos >= max(2, int(0.7 * len(rows))):
        rows.sort(key=lambda r: (
            np.inf if not np.isfinite(r.get("position", np.nan)) else r["position"],
            r["filename"]
        ))
        sort_mode = "ImagePositionPatient"
    elif valid_inst >= max(2, int(0.7 * len(rows))):
        rows.sort(key=lambda r: (
            np.inf if not np.isfinite(r.get("instance", np.nan)) else r["instance"],
            r["filename"]
        ))
        sort_mode = "InstanceNumber"
    else:
        def natural_key(name):
            return [int(part) if part.isdigit() else part.lower()
                    for part in re.split(r"(\d+)", name)]
        rows.sort(key=lambda r: natural_key(r["filename"]))
        sort_mode = "filename_fallback"

    for i, r in enumerate(rows):
        r["sort_index"] = i
        r["sort_mode"] = sort_mode
    return rows


def decode_dicom_slice(path: Path):
    ds = pydicom.dcmread(str(path), force=True)
    pixels = np.asarray(ds.pixel_array)

    if pixels.ndim == 3:
        if pixels.shape[-1] in (3, 4):
            pixels = pixels[..., :3].astype(np.float32)
            pixels = 0.299 * pixels[..., 0] + 0.587 * pixels[..., 1] + 0.114 * pixels[..., 2]
        else:
            pixels = pixels[pixels.shape[0] // 2]

    pixels = np.squeeze(pixels)
    if pixels.ndim != 2:
        raise ValueError(f"Expected 2D MRI slice, got shape={pixels.shape}")

    pixels = pixels.astype(np.float32)
    try:
        pixels = np.asarray(apply_modality_lut(pixels, ds), dtype=np.float32)
    except Exception:
        slope = _safe_float(getattr(ds, "RescaleSlope", 1.0), 1.0)
        intercept = _safe_float(getattr(ds, "RescaleIntercept", 0.0), 0.0)
        pixels = pixels * slope + intercept

    if str(getattr(ds, "PhotometricInterpretation", "MONOCHROME2")).upper() == "MONOCHROME1":
        lo, hi = np.nanmin(pixels), np.nanmax(pixels)
        pixels = hi + lo - pixels

    pixels[~np.isfinite(pixels)] = 0
    return pixels


def normalize_resize_series(image_list, output_size=IMG_SIZE):
    if not image_list:
        return []
    all_values = np.concatenate([img.reshape(-1) for img in image_list])
    lo, hi = np.percentile(all_values, [1.0, 99.5])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.min(all_values)), float(np.max(all_values))
    if hi <= lo:
        hi = lo + 1.0

    output = []
    for img in image_list:
        x = np.clip(img, lo, hi)
        x = (x - lo) / (hi - lo)
        x = np.clip(x, 0, 1)
        x = (x * 255.0).round().astype(np.uint8)
        x = cv2.resize(x, (output_size, output_size), interpolation=cv2.INTER_AREA)
        output.append(x)
    return output


def process_one_test_series(series_row):
    uid = str(series_row[ID_COL])
    series_uid = str(series_row["SeriesInstanceUID"])
    plane = str(series_row["Anatomical_Plane"])
    series_dir = TEST_SERIES_DIR / uid / series_uid
    if not series_dir.exists():
        return [], [], f"missing_series_directory: {series_dir}"

    headers = read_sorted_dicom_headers(series_dir)
    if not headers:
        return [], [], "no_dicom_files"

    n = len(headers)
    target_n = min(SLICES_PER_SERIES, n)
    initial = np.unique(np.linspace(0, n - 1, target_n).round().astype(int)).tolist()
    initial_set = set(initial)
    candidates = initial + [i for i in range(n) if i not in initial_set]

    decoded, decoded_positions, errors = [], [], []
    for idx in candidates:
        try:
            img = decode_dicom_slice(headers[idx]["path"])
            decoded.append(img)
            decoded_positions.append(idx / max(n - 1, 1))
            if len(decoded) >= SLICES_PER_SERIES:
                break
        except Exception as exc:
            errors.append(f"{headers[idx]['filename']}: {type(exc).__name__}: {exc}")

    if len(decoded) < MIN_DECODED_SLICES_PER_SERIES:
        return [], [], "too_few_decoded_slices: " + " | ".join(errors[:2])

    order = np.argsort(decoded_positions)
    decoded = [decoded[i] for i in order]
    decoded_positions = [decoded_positions[i] for i in order]
    normalized = normalize_resize_series(decoded)

    plane_id = PLANE_TO_ID.get(plane, -1)
    fluid_value = _safe_float(series_row.get("Fluid_Sensitive", -1), -1)
    fat_value = _safe_float(series_row.get("Fat_Suppression", -1), -1)
    fluid = int(fluid_value) if fluid_value in (0, 1) else -1
    fat = int(fat_value) if fat_value in (0, 1) else -1

    metadata = [
        {"plane_id": plane_id, "fluid": fluid, "fat": fat, "position": float(pos)}
        for pos in decoded_positions
    ]
    return normalized, metadata, "ok"


def process_one_test_study(uid):
    image_list, metadata_list, used_series, errors = [], [], [], []
    selected = sorted(selected_by_uid.get(str(uid), []), key=lambda x: x["plane_id"])

    for item in selected:
        try:
            imgs, metas, status = process_one_test_series(pd.Series(item))
        except Exception as exc:
            imgs, metas = [], []
            status = f"exception:{type(exc).__name__}:{exc}"

        if status == "ok":
            image_list.extend(imgs)
            metadata_list.extend(metas)
            used_series.append(item["SeriesInstanceUID"])
        else:
            errors.append(f"{item['Anatomical_Plane']}:{item['SeriesInstanceUID']}:{status}")

    if not image_list:
        return None, {
            "status": "failed",
            "error": "; ".join(errors) or "no selected series",
            "n_slices": 0,
            "n_series_used": 0,
        }

    if len(image_list) > MAX_TOTAL_SLICES:
        image_list = image_list[:MAX_TOTAL_SLICES]
        metadata_list = metadata_list[:MAX_TOTAL_SLICES]

    images = np.zeros((MAX_TOTAL_SLICES, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    plane_ids = np.full(MAX_TOTAL_SLICES, -1, dtype=np.int8)
    fluid_flags = np.full(MAX_TOTAL_SLICES, -1, dtype=np.int8)
    fat_flags = np.full(MAX_TOTAL_SLICES, -1, dtype=np.int8)
    positions = np.zeros(MAX_TOTAL_SLICES, dtype=np.float32)
    valid = np.zeros(MAX_TOTAL_SLICES, dtype=np.uint8)

    n_valid = len(image_list)
    images[:n_valid] = np.stack(image_list, axis=0)
    valid[:n_valid] = 1

    for i, meta in enumerate(metadata_list):
        plane_ids[i] = meta["plane_id"]
        fluid_flags[i] = meta["fluid"]
        fat_flags[i] = meta["fat"]
        positions[i] = meta["position"]

    return (images, plane_ids, fluid_flags, fat_flags, positions, valid), {
        "status": "ok",
        "error": "; ".join(errors),
        "n_slices": n_valid,
        "n_series_used": len(used_series),
        "used_series": ";".join(used_series),
    }

print("Test DICOM preprocessing functions are ready.")

Test DICOM preprocessing functions are ready.


## 6. ViT-B/16: resumable test cache

In [24]:
N_TEST = len(test_df)
T = MAX_TOTAL_SLICES

volume_path = CACHE_DIR / "test_slices_uint8.npy"
valid_path = CACHE_DIR / "slice_valid.npy"
plane_path = CACHE_DIR / "plane_ids.npy"
fluid_path = CACHE_DIR / "fluid_flags.npy"
fat_path = CACHE_DIR / "fat_flags.npy"
position_path = CACHE_DIR / "slice_positions.npy"
status_path = CACHE_DIR / "test_preprocess_status.csv"

required_cache = [volume_path, valid_path, plane_path, fluid_path, fat_path, position_path]
cache_exists = all(p.exists() for p in required_cache)

if cache_exists:
    existing = np.load(volume_path, mmap_mode="r")
    if existing.shape != (N_TEST, T, IMG_SIZE, IMG_SIZE) or existing.dtype != np.uint8:
        print(
            f"Stale cache detected ({existing.shape}/{existing.dtype}) for N_TEST={N_TEST}; "
            "recreating cache for the current test set."
        )
        del existing
        for p in required_cache:
            if p.exists():
                p.unlink()
        if status_path.exists():
            status_path.unlink()
        cache_exists = False
    else:
        del existing

if cache_exists:
    volumes = np.lib.format.open_memmap(volume_path, mode="r+")
    valid_mm = np.lib.format.open_memmap(valid_path, mode="r+")
    plane_mm = np.lib.format.open_memmap(plane_path, mode="r+")
    fluid_mm = np.lib.format.open_memmap(fluid_path, mode="r+")
    fat_mm = np.lib.format.open_memmap(fat_path, mode="r+")
    position_mm = np.lib.format.open_memmap(position_path, mode="r+")
    print("Resuming existing test cache.")
else:
    required_gb = N_TEST * T * IMG_SIZE * IMG_SIZE / 1024**3
    free_gb = shutil.disk_usage(OUTPUT_ROOT).free / 1024**3
    print(f"Estimated test image cache: {required_gb:.2f} GiB | Free output disk: {free_gb:.2f} GiB")
    if free_gb < required_gb + 1.0:
        raise RuntimeError("Not enough output disk space for test cache.")

    volumes = np.lib.format.open_memmap(
        volume_path, mode="w+", dtype=np.uint8,
        shape=(N_TEST, T, IMG_SIZE, IMG_SIZE)
    )
    valid_mm = np.lib.format.open_memmap(valid_path, mode="w+", dtype=np.uint8, shape=(N_TEST, T))
    plane_mm = np.lib.format.open_memmap(plane_path, mode="w+", dtype=np.int8, shape=(N_TEST, T))
    fluid_mm = np.lib.format.open_memmap(fluid_path, mode="w+", dtype=np.int8, shape=(N_TEST, T))
    fat_mm = np.lib.format.open_memmap(fat_path, mode="w+", dtype=np.int8, shape=(N_TEST, T))
    position_mm = np.lib.format.open_memmap(position_path, mode="w+", dtype=np.float32, shape=(N_TEST, T))

    volumes[:] = 0
    valid_mm[:] = 0
    plane_mm[:] = -1
    fluid_mm[:] = -1
    fat_mm[:] = -1
    position_mm[:] = 0
    for arr in [volumes, valid_mm, plane_mm, fluid_mm, fat_mm, position_mm]:
        arr.flush()

if status_path.exists():
    prior = pd.read_csv(status_path).fillna("")
    status_by_uid = {str(r[ID_COL]): r.to_dict() for _, r in prior.iterrows()}
else:
    status_by_uid = {}

print("Test studies:", N_TEST)
print("Completed test preprocessing records:", len(status_by_uid))
print("Cache directory:", CACHE_DIR)

Resuming existing test cache.
Test studies: 3
Completed test preprocessing records: 3
Cache directory: /kaggle/working/model_ensemble_v2/vit_inference/test_preprocessed_cache


## 7. ViT-B/16: preprocess test studies

In [25]:
def run_test_preprocess_batch(start_idx, end_idx, batch_name):
    end_idx = min(end_idx, N_TEST)
    if start_idx >= end_idx:
        print(f"{batch_name}: no studies in this range ({start_idx}:{end_idx}).")
        return

    started = time.time()
    processed_now = 0
    uid_list = test_df[ID_COL].astype(str).tolist()

    print(f"{batch_name}: studies [{start_idx}, {end_idx}) / {N_TEST}")

    for row_idx in range(start_idx, end_idx):
        uid = uid_list[row_idx]
        if status_by_uid.get(uid, {}).get("status") == "ok":
            continue

        result, info = process_one_test_study(uid)

        if result is not None:
            images, plane_ids, fluid, fat, positions, valid = result
            volumes[row_idx] = images
            plane_mm[row_idx] = plane_ids
            fluid_mm[row_idx] = fluid
            fat_mm[row_idx] = fat
            position_mm[row_idx] = positions
            valid_mm[row_idx] = valid
        else:
            volumes[row_idx] = 0
            plane_mm[row_idx] = -1
            fluid_mm[row_idx] = -1
            fat_mm[row_idx] = -1
            position_mm[row_idx] = 0
            valid_mm[row_idx] = 0

        status_by_uid[uid] = {
            ID_COL: uid,
            "row_idx": row_idx,
            "status": info.get("status", "failed"),
            "n_slices": info.get("n_slices", 0),
            "n_series_used": info.get("n_series_used", 0),
            "error": info.get("error", ""),
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        }

        processed_now += 1
        if processed_now % 5 == 0 or row_idx + 1 == end_idx:
            done = sum(v.get("status") == "ok" for v in status_by_uid.values())
            print(
                f"[{row_idx + 1}/{N_TEST}] overall_ok={done}/{N_TEST} "
                f"current_status={info.get('status')} "
                f"elapsed={(time.time()-started)/60:.1f} min",
                flush=True,
            )

        if processed_now % PREPROCESS_SAVE_EVERY == 0:
            for arr in [volumes, valid_mm, plane_mm, fluid_mm, fat_mm, position_mm]:
                arr.flush()
            pd.DataFrame(list(status_by_uid.values())).to_csv(status_path, index=False)

    for arr in [volumes, valid_mm, plane_mm, fluid_mm, fat_mm, position_mm]:
        arr.flush()
    pd.DataFrame(list(status_by_uid.values())).to_csv(status_path, index=False)
    print(f"{batch_name} completed. Newly processed: {processed_now}")


run_test_preprocess_batch(0, N_TEST, "VIT TEST PREPROCESS")


VIT TEST PREPROCESS: studies [0, 3) / 3
VIT TEST PREPROCESS completed. Newly processed: 0


## 8. ViT-B/16: audit preprocessing

In [26]:
status_df = pd.read_csv(status_path).fillna("")
status_df[ID_COL] = status_df[ID_COL].astype(str)
status_by_uid_final = set(status_df[ID_COL])
expected_test_ids = set(test_df[ID_COL].astype(str))

missing_status = expected_test_ids - status_by_uid_final
failed_preprocess = status_df[status_df["status"].ne("ok")].copy()

print("Expected test studies:", len(expected_test_ids))
print("Status records:", len(status_df))
print("Successful preprocessing:", int(status_df["status"].eq("ok").sum()))
print("Failed preprocessing:", len(failed_preprocess))
print("Missing status records:", len(missing_status))

if missing_status:
    raise RuntimeError(f"Missing preprocessing records for {len(missing_status)} test studies.")

if len(failed_preprocess):
    print("Failed studies will receive neutral 0.5 fallback predictions to keep the required submission schema.")
    display(failed_preprocess[[ID_COL, "n_slices", "error"]].head(20))

status_df.to_csv(OUTPUT_ROOT / "test_preprocess_status.csv", index=False)

Expected test studies: 3
Status records: 3
Successful preprocessing: 3
Failed preprocessing: 0
Missing status records: 0


## 9. ViT-B/16: restore checkpoint architecture

In [27]:
# Load checkpoint metadata first. The checkpoint contains the full fine-tuned ViT-B/16 weights,
# so torchvision is instantiated with weights=None (no network/download required).
try:
    checkpoint = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
except TypeError:
    checkpoint = torch.load(BEST_CKPT, map_location="cpu")

if checkpoint.get("targets") != TARGETS:
    raise ValueError(
        "Checkpoint target order does not match competition sample submission.\n"
        f"Checkpoint={checkpoint.get('targets')}\nExpected={TARGETS}"
    )
if "model_state" not in checkpoint:
    raise KeyError("Checkpoint does not contain model_state")

run_config = checkpoint.get("run_config", {})
IMAGE_SIZE = int(run_config.get("image_size", IMG_SIZE))
PATCH_SIZE = int(run_config.get("patch_size", PATCH_SIZE))
EMBED_DIM = int(run_config.get("embed_dim", 256))
MAX_SLICES = int(run_config.get("max_slices", MAX_TOTAL_SLICES))
STUDY_DEPTH = int(run_config.get("study_depth", 2))
STUDY_HEADS = int(run_config.get("study_heads", 4))
FREEZE_EARLY_BLOCKS = int(run_config.get("freeze_early_blocks", 10))
SLICE_CHUNK_SIZE = int(run_config.get("slice_chunk_size", SLICE_CHUNK_SIZE))
USE_CENTER_PRIOR = bool(run_config.get("use_center_prior", True))
CENTER_PRIOR_STRENGTH = float(run_config.get("center_prior_strength", 0.15))
CENTER_PRIOR_SIGMA = float(run_config.get("center_prior_sigma", 0.55))

if IMAGE_SIZE != IMG_SIZE or MAX_SLICES != MAX_TOTAL_SLICES:
    raise ValueError(
        "Submission preprocessing dimensions do not match checkpoint config: "
        f"checkpoint image_size={IMAGE_SIZE}, max_slices={MAX_SLICES}; "
        f"submission image_size={IMG_SIZE}, max_slices={MAX_TOTAL_SLICES}"
    )
if PATCH_SIZE != 16:
    raise ValueError(f"Expected ViT-B/16 patch size 16, got {PATCH_SIZE}")

print("Checkpoint best epoch:", checkpoint.get("best_epoch"))
print("Checkpoint validation BCE:", checkpoint.get("best_val_loss"))
print("Checkpoint architecture:", run_config.get("architecture", "not recorded"))
print("Model config:", {
    "image_size": IMAGE_SIZE,
    "patch_size": PATCH_SIZE,
    "embed_dim": EMBED_DIM,
    "study_depth": STUDY_DEPTH,
    "study_heads": STUDY_HEADS,
    "center_prior": USE_CENTER_PRIOR,
    "center_prior_strength": CENTER_PRIOR_STRENGTH,
    "center_prior_sigma": CENTER_PRIOR_SIGMA,
    "slice_chunk_size": SLICE_CHUNK_SIZE,
})


class PretrainedSliceEncoder(nn.Module):
    def __init__(
        self,
        backbone,
        output_dim=EMBED_DIM,
        freeze_early_blocks=FREEZE_EARLY_BLOCKS,
        use_center_prior=USE_CENTER_PRIOR,
        center_prior_strength=CENTER_PRIOR_STRENGTH,
        center_prior_sigma=CENTER_PRIOR_SIGMA,
        dropout=0.10,
    ):
        super().__init__()
        self.backbone = backbone
        self.hidden_dim = 768
        self.freeze_early_blocks = int(freeze_early_blocks)
        self.use_center_prior = bool(use_center_prior)

        blocks = list(self.backbone.encoder.layers.children())
        if not 0 <= self.freeze_early_blocks < len(blocks):
            raise ValueError(
                f"freeze_early_blocks must be in [0, {len(blocks)-1}]"
            )

        for p in self.backbone.parameters():
            p.requires_grad = False
        for block in blocks[self.freeze_early_blocks:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in self.backbone.encoder.ln.parameters():
            p.requires_grad = True

        self.trainable_blocks = blocks[self.freeze_early_blocks:]
        self.patch_attention = nn.Sequential(
            nn.LayerNorm(self.hidden_dim),
            nn.Linear(self.hidden_dim, 1, bias=False),
        )
        self.feature_projection = nn.Sequential(
            nn.Linear(self.hidden_dim * 2, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        grid = IMAGE_SIZE // PATCH_SIZE
        coords = torch.linspace(-1.0, 1.0, grid)
        yy, xx = torch.meshgrid(coords, coords, indexing="ij")
        sigma = max(float(center_prior_sigma), 1e-4)
        log_gaussian = -0.5 * ((xx / sigma) ** 2 + (yy / sigma) ** 2)
        log_gaussian = log_gaussian - log_gaussian.max()
        strength = float(center_prior_strength) if use_center_prior else 0.0
        self.register_buffer(
            "center_bias",
            (strength * log_gaussian.reshape(-1)).float(),
        )

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()
        for block in self.trainable_blocks:
            block.train(mode)
        self.backbone.encoder.ln.train(mode)
        self.patch_attention.train(mode)
        self.feature_projection.train(mode)
        return self

    def forward(self, images_rgb):
        # images_rgb: [N, 3, 224, 224], already ImageNet-normalized.
        x = self.backbone._process_input(images_rgb)
        cls = self.backbone.class_token.expand(x.shape[0], -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.backbone.encoder.pos_embedding
        x = self.backbone.encoder.dropout(x)

        for block_i, block in enumerate(self.backbone.encoder.layers.children()):
            if block_i < self.freeze_early_blocks:
                with torch.no_grad():
                    x = block(x)
            else:
                x = block(x)

        x = self.backbone.encoder.ln(x)
        cls_feature = x[:, 0]
        patch_tokens = x[:, 1:]

        patch_scores = self.patch_attention(patch_tokens).squeeze(-1)
        patch_scores = patch_scores + self.center_bias.unsqueeze(0)
        patch_weights = torch.softmax(patch_scores, dim=1)
        spatial_feature = torch.sum(
            patch_tokens * patch_weights.unsqueeze(-1), dim=1
        )

        return self.feature_projection(
            torch.cat([cls_feature, spatial_feature], dim=-1)
        )


class StudyFeatureAggregator(nn.Module):
    """Gated metadata + study Transformer + residual mean/attention pooling."""

    def __init__(
        self,
        embed_dim=EMBED_DIM,
        max_slices=MAX_SLICES,
        study_depth=STUDY_DEPTH,
        num_heads=STUDY_HEADS,
        dropout=0.10,
    ):
        super().__init__()
        self.plane_embedding = nn.Embedding(4, embed_dim, padding_idx=3)
        self.metadata_projection = nn.Sequential(
            nn.Linear(3, embed_dim),
            nn.GELU(),
            nn.Linear(embed_dim, embed_dim),
        )
        self.slice_order_embedding = nn.Embedding(max_slices, embed_dim)
        self.metadata_scale = nn.Parameter(torch.tensor(0.10))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.study_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=study_depth,
            enable_nested_tensor=False,
        )

        self.attention_score = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 1),
        )
        self.pool_projection = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.skip_scale = nn.Parameter(torch.tensor(0.5))
        self.output_norm = nn.LayerNorm(embed_dim)

    def forward(
        self,
        slice_features,
        valid,
        plane_ids,
        fluid,
        fat,
        positions,
    ):
        batch_size, num_slices, embed_dim = slice_features.shape
        valid = valid.bool()

        if not valid.any(dim=1).all():
            raise ValueError(
                "Every study must contain at least one valid slice."
            )

        plane_ids = plane_ids.clamp(0, 3)
        metadata = torch.stack([
            fluid.clamp(-1, 1),
            fat.clamp(-1, 1),
            positions.clamp(0, 1),
        ], dim=-1)
        order_ids = torch.arange(
            num_slices, device=slice_features.device
        ).unsqueeze(0)

        metadata_features = (
            self.plane_embedding(plane_ids)
            + self.metadata_projection(metadata)
            + self.slice_order_embedding(order_ids)
        )
        tokens = slice_features + self.metadata_scale * metadata_features

        encoded = self.study_encoder(
            tokens,
            src_key_padding_mask=~valid,
        )

        mask = valid.unsqueeze(-1).to(encoded.dtype)
        denominator = mask.sum(dim=1).clamp_min(1.0)

        mean_pool = (encoded * mask).sum(dim=1) / denominator

        scores = self.attention_score(encoded).squeeze(-1)
        scores = scores.masked_fill(
            ~valid, torch.finfo(scores.dtype).min
        )
        attention_weights = torch.softmax(scores, dim=1)
        attention_pool = torch.sum(
            encoded * attention_weights.unsqueeze(-1), dim=1
        )

        pooled = self.pool_projection(
            torch.cat([mean_pool, attention_pool], dim=-1)
        )

        raw_mean = (
            slice_features * mask
        ).sum(dim=1) / denominator

        return self.output_norm(
            pooled + self.skip_scale * raw_mean
        )


class PretrainedViTStudyModel(nn.Module):
    def __init__(self, backbone, num_targets=len(TARGETS)):
        super().__init__()
        self.slice_encoder = PretrainedSliceEncoder(backbone=backbone)
        self.aggregator = StudyFeatureAggregator()
        self.classifier = nn.Sequential(
            nn.Linear(EMBED_DIM, 128),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(128, num_targets),
        )
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        self.register_buffer("image_mean", mean)
        self.register_buffer("image_std", std)

    def forward(self, images, valid, plane_ids, fluid, fat, positions):
        batch_size, num_slices, height, width = images.shape
        if height != IMAGE_SIZE or width != IMAGE_SIZE:
            raise ValueError(
                f"Expected {IMAGE_SIZE}x{IMAGE_SIZE}; got {height}x{width}"
            )

        valid_bool = valid.bool()
        valid_flat = valid_bool.reshape(-1)
        valid_indices = valid_flat.nonzero(as_tuple=False).squeeze(1)
        if valid_indices.numel() == 0:
            raise ValueError("Batch contains no valid MRI slices")

        flat = images.reshape(
            batch_size * num_slices, 1, height, width
        )
        selected = flat.index_select(
            0, valid_indices
        ).float().div(255.0)
        selected = selected.repeat(1, 3, 1, 1)
        selected = (selected - self.image_mean) / self.image_std

        encoded_chunks = []
        for start in range(0, selected.shape[0], SLICE_CHUNK_SIZE):
            chunk = selected[start:start + SLICE_CHUNK_SIZE]
            encoded_chunks.append(self.slice_encoder(chunk))
        valid_features = torch.cat(encoded_chunks, dim=0)

        flat_features = valid_features.new_zeros(
            (batch_size * num_slices, EMBED_DIM)
        )
        flat_features = flat_features.index_copy(
            0, valid_indices, valid_features
        )
        slice_features = flat_features.view(
            batch_size, num_slices, EMBED_DIM
        )

        study_features = self.aggregator(
            slice_features,
            valid_bool,
            plane_ids,
            fluid,
            fat,
            positions,
        )
        return self.classifier(study_features)


# Build the exact architecture without downloading pretrained weights.
backbone = vit_b_16(weights=None)
model = PretrainedViTStudyModel(backbone=backbone).to(device)

try:
    incompatible = model.load_state_dict(
        checkpoint["model_state"], strict=True
    )
except RuntimeError as exc:
    raise RuntimeError(
        "Checkpoint không tương thích với Residual Attention Pooling model. "
        "Hãy chắc chắn bạn đã Add Input output của version training mới, "
        "không phải artifact ViT Study-CLS cũ. Chi tiết:\n" + str(exc)
    ) from exc

model.eval()
print("Residual Attention Pooling ViT checkpoint loaded successfully.")
print("Best epoch:", checkpoint.get("best_epoch"))
print("Best validation BCE:", checkpoint.get("best_val_loss"))
print("Model parameters:", f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print("Device:", device)


# Close the archive stream after the checkpoint has been loaded into CPU memory.
if "VIT_ZIP_HANDLE" in globals():
    try:
        BEST_CKPT.close()
        VIT_ZIP_HANDLE.close()
    except Exception:
        pass


Checkpoint best epoch: 7
Checkpoint validation BCE: 0.6440770427385966
Checkpoint architecture: Pretrained ViT-B/16 + gated metadata + study Transformer + residual mean-attention pooling
Model config: {'image_size': 224, 'patch_size': 16, 'embed_dim': 256, 'study_depth': 2, 'study_heads': 4, 'center_prior': True, 'center_prior_strength': 0.15, 'center_prior_sigma': 0.55, 'slice_chunk_size': 8}
Residual Attention Pooling ViT checkpoint loaded successfully.
Best epoch: 7
Best validation BCE: 0.6440770427385966
Model parameters: 88.79M
Device: cuda


## 10. ViT-B/16: inference on test studies

In [28]:
# Read-only memory maps for test cache.
volumes = np.load(volume_path, mmap_mode="r")
valid_cache = np.load(valid_path, mmap_mode="r")
plane_cache = np.load(plane_path, mmap_mode="r")
fluid_cache = np.load(fluid_path, mmap_mode="r")
fat_cache = np.load(fat_path, mmap_mode="r")
position_cache = np.load(position_path, mmap_mode="r")

uid_list = test_df[ID_COL].astype(str).tolist()
uid_to_idx = {uid: i for i, uid in enumerate(uid_list)}

class TestStudyDataset(Dataset):
    def __init__(self, indices):
        self.indices = list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        idx = int(self.indices[item])
        return {
            "images": torch.from_numpy(np.array(volumes[idx], copy=True)),
            "valid": torch.from_numpy(np.array(valid_cache[idx], copy=True).astype(np.uint8)),
            "plane_ids": torch.from_numpy(np.array(plane_cache[idx], copy=True).astype(np.int64)),
            "fluid": torch.from_numpy(np.array(fluid_cache[idx], copy=True).astype(np.float32)),
            "fat": torch.from_numpy(np.array(fat_cache[idx], copy=True).astype(np.float32)),
            "positions": torch.from_numpy(np.array(position_cache[idx], copy=True).astype(np.float32)),
            "uid": uid_list[idx],
        }


# Only pass studies with at least one successfully decoded slice through the network.
# Failed DICOM studies retain neutral 0.5 fallback predictions.
valid_indices = [i for i in range(N_TEST) if int(valid_cache[i].sum()) > 0]
failed_indices = sorted(set(range(N_TEST)) - set(valid_indices))

print("Studies with usable DICOM cache:", len(valid_indices))
print("Studies with no usable DICOM slices:", len(failed_indices))

pred_by_uid = {
    uid: np.full(len(TARGETS), 0.5, dtype=np.float32)
    for uid in uid_list
}

inference_dataset = TestStudyDataset(valid_indices)
inference_loader = DataLoader(
    inference_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

model.eval()
started = time.time()

with torch.inference_mode():
    for step, batch in enumerate(inference_loader, start=1):
        images = batch["images"].to(device, non_blocking=True)
        valid = batch["valid"].to(device, non_blocking=True)
        plane_ids = batch["plane_ids"].to(device, non_blocking=True)
        fluid = batch["fluid"].to(device, non_blocking=True)
        fat = batch["fat"].to(device, non_blocking=True)
        positions = batch["positions"].to(device, non_blocking=True)

        logits = model(images, valid, plane_ids, fluid, fat, positions)
        probs = torch.sigmoid(logits.float()).cpu().numpy()

        for uid, prob in zip(batch["uid"], probs):
            pred_by_uid[str(uid)] = prob.astype(np.float32)

        if step % 25 == 0 or step == len(inference_loader):
            print(
                f"Inference batch {step}/{len(inference_loader)} "
                f"| studies={min(step*BATCH_SIZE, len(valid_indices))}/{len(valid_indices)} "
                f"| elapsed={(time.time()-started)/60:.1f} min",
                flush=True,
            )

print("Inference finished in", round((time.time() - started) / 60, 2), "minutes")

# Preserve predictions and per-study success flags for blending.
vit_probabilities = np.stack([pred_by_uid[uid] for uid in test_uids]).astype(np.float32)
vit_valid_mask = np.zeros(N_TEST, dtype=bool)
vit_valid_mask[np.asarray(valid_indices, dtype=np.int64)] = True
if vit_probabilities.shape != (N_TEST, len(TARGETS)):
    raise RuntimeError(f"ViT prediction shape mismatch: {vit_probabilities.shape}")
print("ViT inference output:", vit_probabilities.shape, "| valid studies:", int(vit_valid_mask.sum()), "/", N_TEST)

# Release ViT GPU memory before loading the five DINOv2 fold models.
del model, backbone, checkpoint, inference_loader, inference_dataset
gc.collect()
torch.cuda.empty_cache()
print("ViT model released. GPU allocated GiB:", round(torch.cuda.memory_allocated()/1024**3, 2))


# Preserve the ViT probabilities and indicate which studies had a usable image cache.
vit_probabilities = np.stack([pred_by_uid[uid] for uid in test_uids]).astype(np.float32)
vit_valid_mask = np.zeros(N_TEST, dtype=bool)
vit_valid_mask[np.asarray(valid_indices, dtype=np.int64)] = True
if vit_probabilities.shape != (N_TEST, len(TARGETS)):
    raise RuntimeError(f"ViT prediction shape mismatch: {vit_probabilities.shape}")
print(
    "ViT predictions:", vit_probabilities.shape,
    "| usable studies:", int(vit_valid_mask.sum()), "/", N_TEST,
)


Studies with usable DICOM cache: 3
Studies with no usable DICOM slices: 0
Inference batch 3/3 | studies=3/3 | elapsed=0.0 min
Inference finished in 0.02 minutes
ViT inference output: (3, 12) | valid studies: 3 / 3
ViT model released. GPU allocated GiB: 0.01
ViT predictions: (3, 12) | usable studies: 3 / 3


## 11. Locate DINOv2 5-fold model bundle

Cell hỗ trợ cả Dataset đã giải nén lẫn ZIP. Notebook ưu tiên đường dẫn cấu hình ở Cell 1; nếu đường dẫn mount khác trong phiên submit, nó tự tìm bundle hợp lệ trong `/kaggle/input`. Với Dataset đã giải nén, notebook đọc trực tiếp 5 checkpoint và chỉ sử dụng mã nguồn `dinov2_repo/` tại chỗ; không copy 5 checkpoint lớn sang `/kaggle/working`.

In [29]:
# ============================================================
# DINOv2 artifact loader — supports extracted Dataset and ZIP
# ============================================================
DINO_BUNDLE_NAME = "five_role_v2_cv5_submission_bundle.zip"
DINO_MANIFEST_NAME = "bundle_manifest.json"
DINO_REPO_NAME = "dinov2_repo"
DINO_REPO_HUBCONF = "dinov2_repo/hubconf.py"
DINO_FOLD_RELATIVE_MEMBERS = [
    f"five_role_2p5d_v2_cv5/fold_{fold}/best_checkpoint.pt"
    for fold in range(1, 6)
]


class _DirectoryBundleReader:
    """File-reader with the same .open()/.close() interface used by Cell 31."""
    def __init__(self, root):
        self.root = Path(root).resolve()

    def open(self, member, mode="r"):
        # torch.load uses a binary, seekable stream.
        relative = Path(str(member).replace("\\", "/"))
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError(f"Unsafe DINOv2 bundle member: {member}")
        path = (self.root / relative).resolve()
        if self.root != path and self.root not in path.parents:
            raise RuntimeError(f"DINOv2 member escapes bundle root: {member}")
        if not path.is_file():
            raise FileNotFoundError(f"DINOv2 bundle member not found: {path}")
        return path.open("rb")

    def close(self):
        # Each .open() call returns its own file handle, which is closed by `with`.
        pass


def _is_valid_dino_directory(root):
    root = Path(root)
    required = [
        root / DINO_MANIFEST_NAME,
        root / DINO_REPO_NAME / "hubconf.py",
        root / DINO_REPO_NAME / "dinov2",
        *[root / member for member in DINO_FOLD_RELATIVE_MEMBERS],
    ]
    return (
        all(path.exists() for path in required)
        and (root / DINO_MANIFEST_NAME).is_file()
        and (root / DINO_REPO_NAME / "hubconf.py").is_file()
        and all((root / member).is_file() for member in DINO_FOLD_RELATIVE_MEMBERS)
    )


def _find_valid_dino_root(dataset_path):
    dataset_path = Path(dataset_path)
    manifest_paths = []
    direct_manifest = dataset_path / DINO_MANIFEST_NAME
    if direct_manifest.is_file():
        manifest_paths.append(direct_manifest)
    else:
        manifest_paths = list(dataset_path.rglob(DINO_MANIFEST_NAME))

    valid_roots = []
    for manifest_path in manifest_paths:
        candidate_root = manifest_path.parent
        if _is_valid_dino_directory(candidate_root):
            valid_roots.append(candidate_root.resolve())

    # De-duplicate while preserving order.
    valid_roots = list(dict.fromkeys(valid_roots))
    if len(valid_roots) != 1:
        details = "\n".join(str(p) for p in valid_roots) or "(none)"
        raise RuntimeError(
            "Expected exactly one complete extracted DINOv2 bundle inside "
            f"{dataset_path}; found {len(valid_roots)}:\n{details}\n"
            "Expected bundle_manifest.json, dinov2_repo/hubconf.py, and "
            "five fold checkpoints under five_role_2p5d_v2_cv5/."
        )
    return valid_roots[0]


if DINO_BUNDLE_OVERRIDE and Path(DINO_BUNDLE_OVERRIDE).exists():
    dino_bundle_path = Path(DINO_BUNDLE_OVERRIDE)
else:
    # Kaggle may expose a Dataset at a slightly different mount path in a
    # committed run. Search attached inputs for either an extracted bundle
    # (identified by its manifest + all five folds) or the original ZIP.
    discovered = []
    for manifest_path in INPUT_ROOT.rglob(DINO_MANIFEST_NAME):
        candidate_root = manifest_path.parent
        if _is_valid_dino_directory(candidate_root):
            discovered.append(candidate_root.resolve())

    zip_candidates = find_named_files([INPUT_ROOT], DINO_BUNDLE_NAME)
    discovered.extend(zip_candidates)
    discovered = list({str(p.resolve()): p for p in discovered}.values())

    if len(discovered) != 1:
        details = "\n".join(str(p) for p in discovered) or "(none)"
        raise RuntimeError(
            "Could not uniquely locate the DINOv2 5-fold bundle in attached "
            f"Kaggle inputs; found {len(discovered)} candidates:\n{details}\n"
            f"Configured override: {DINO_BUNDLE_OVERRIDE or '(empty)'}\n"
            "Attach the DINOv2 Dataset to this notebook version and check that "
            "it contains bundle_manifest.json plus all five fold checkpoints."
        )

    dino_bundle_path = discovered[0]
    print("DINOv2 override path was unavailable or empty; auto-discovered:", dino_bundle_path)

DINO_ZIP_HANDLE = None
DINO_BUNDLE_IS_ZIP = False
DINO_BUNDLE_ROOT = None
DINO_BUNDLE_PREFIX = ""

if dino_bundle_path.is_dir():
    # --------------------------------------------------------
    # Case A: Kaggle Dataset is already extracted into files.
    # --------------------------------------------------------
    DINO_BUNDLE_ROOT = _find_valid_dino_root(dino_bundle_path)
    DINOV2_REPO_DIR = DINO_BUNDLE_ROOT / DINO_REPO_NAME
    DINO_FOLD_MEMBERS = list(DINO_FOLD_RELATIVE_MEMBERS)

    bundle_manifest = json.loads(
        (DINO_BUNDLE_ROOT / DINO_MANIFEST_NAME).read_text(encoding="utf-8")
    )
    DINO_ZIP_HANDLE = _DirectoryBundleReader(DINO_BUNDLE_ROOT)

elif dino_bundle_path.is_file() and dino_bundle_path.suffix.lower() == ".zip":
    # --------------------------------------------------------
    # Case B: bundle is still a ZIP archive.
    # A single enclosing folder inside the ZIP is also supported.
    # --------------------------------------------------------
    DINO_BUNDLE_IS_ZIP = True
    DINO_ZIP_HANDLE = zipfile.ZipFile(dino_bundle_path, "r")
    zip_names = DINO_ZIP_HANDLE.namelist()

    for member in zip_names:
        normalized = member.replace("\\", "/")
        parts = Path(normalized).parts
        if normalized.startswith("/") or ".." in parts:
            DINO_ZIP_HANDLE.close()
            raise RuntimeError(f"Unsafe path in DINOv2 ZIP: {member}")

    manifest_members = [
        name for name in zip_names
        if name.replace("\\", "/").endswith("/" + DINO_MANIFEST_NAME)
        or name.replace("\\", "/") == DINO_MANIFEST_NAME
    ]
    if len(manifest_members) != 1:
        DINO_ZIP_HANDLE.close()
        raise RuntimeError(
            "Expected one bundle_manifest.json in DINOv2 ZIP; "
            f"found {len(manifest_members)}: {manifest_members}"
        )

    manifest_member = manifest_members[0].replace("\\", "/")
    DINO_BUNDLE_PREFIX = manifest_member.rsplit("/", 1)[0] + "/" if "/" in manifest_member else ""
    DINO_FOLD_MEMBERS = [
        DINO_BUNDLE_PREFIX + member for member in DINO_FOLD_RELATIVE_MEMBERS
    ]
    repo_hubconf_member = DINO_BUNDLE_PREFIX + DINO_REPO_HUBCONF

    if repo_hubconf_member not in zip_names:
        DINO_ZIP_HANDLE.close()
        raise FileNotFoundError(
            f"DINOv2 ZIP is missing {repo_hubconf_member}"
        )

    missing_fold_members = [
        member for member in DINO_FOLD_MEMBERS if member not in zip_names
    ]
    if missing_fold_members:
        DINO_ZIP_HANDLE.close()
        raise FileNotFoundError(
            "DINOv2 ZIP is missing fold checkpoints:\n"
            + "\n".join(missing_fold_members)
        )

    # Extract only the local DINOv2 source and manifest, not the 5 large checkpoints.
    DINO_EXTRACT_ROOT = OUTPUT_DIR / "dino_bundle_source"
    if DINO_EXTRACT_ROOT.exists():
        shutil.rmtree(DINO_EXTRACT_ROOT)
    DINO_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

    with DINO_ZIP_HANDLE.open(manifest_member, "r") as src:
        (DINO_EXTRACT_ROOT / DINO_MANIFEST_NAME).write_bytes(src.read())

    repo_prefix = DINO_BUNDLE_PREFIX + DINO_REPO_NAME + "/"
    for member in zip_names:
        normalized = member.replace("\\", "/")
        if not normalized.startswith(repo_prefix) or normalized.endswith("/"):
            continue
        relative_member = normalized[len(DINO_BUNDLE_PREFIX):]
        relative_path = Path(relative_member)
        if relative_path.is_absolute() or ".." in relative_path.parts:
            DINO_ZIP_HANDLE.close()
            raise RuntimeError(f"Unsafe DINOv2 source member: {member}")
        destination = DINO_EXTRACT_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        with DINO_ZIP_HANDLE.open(member, "r") as src, destination.open("wb") as dst:
            shutil.copyfileobj(src, dst)

    DINOV2_REPO_DIR = DINO_EXTRACT_ROOT / DINO_REPO_NAME
    bundle_manifest = json.loads(
        (DINO_EXTRACT_ROOT / DINO_MANIFEST_NAME).read_text(encoding="utf-8")
    )
    DINO_BUNDLE_ROOT = None  # Checkpoints are streamed from the ZIP.

else:
    raise ValueError(
        "DINO_BUNDLE_OVERRIDE must point to an extracted Dataset directory "
        "or a .zip file. Received: "
        f"{dino_bundle_path}"
    )

# Final integrity checks shared by both artifact layouts.
if not (DINOV2_REPO_DIR / "hubconf.py").is_file():
    raise FileNotFoundError(f"Missing local DINOv2 hubconf.py: {DINOV2_REPO_DIR}")
if not (DINOV2_REPO_DIR / "dinov2").is_dir():
    raise FileNotFoundError(f"Missing local DINOv2 package: {DINOV2_REPO_DIR / 'dinov2'}")
if len(DINO_FOLD_MEMBERS) != 5:
    raise RuntimeError(f"Expected five DINOv2 folds, got {len(DINO_FOLD_MEMBERS)}")

print("DINOv2 artifact:", dino_bundle_path)
print("DINOv2 bundle format:", "ZIP" if DINO_BUNDLE_IS_ZIP else "extracted Dataset directory")
print("Resolved bundle root:", DINO_BUNDLE_ROOT if DINO_BUNDLE_ROOT else "inside ZIP")
print("DINOv2 local source:", DINOV2_REPO_DIR)
print("Architecture:", bundle_manifest.get("architecture"))
print("Fold checkpoints:")
for fold, member in enumerate(DINO_FOLD_MEMBERS, start=1):
    print(f"  fold {fold}: {member}")
print("All five checkpoints and local DINOv2 source passed integrity checks.")

DINOv2 artifact: /kaggle/input/datasets/biinseventeen/five-role-v2-cv5-submission-bundle-zip
DINOv2 bundle format: extracted Dataset directory
Resolved bundle root: /kaggle/input/datasets/biinseventeen/five-role-v2-cv5-submission-bundle-zip
DINOv2 local source: /kaggle/input/datasets/biinseventeen/five-role-v2-cv5-submission-bundle-zip/dinov2_repo
Architecture: DINOv2 ViT-S/14 shared 2.5D five-role; target-by-role attention; 12 independent heads
Fold checkpoints:
  fold 1: five_role_2p5d_v2_cv5/fold_1/best_checkpoint.pt
  fold 2: five_role_2p5d_v2_cv5/fold_2/best_checkpoint.pt
  fold 3: five_role_2p5d_v2_cv5/fold_3/best_checkpoint.pt
  fold 4: five_role_2p5d_v2_cv5/fold_4/best_checkpoint.pt
  fold 5: five_role_2p5d_v2_cv5/fold_5/best_checkpoint.pt
All five checkpoints and local DINOv2 source passed integrity checks.


## 12. DINOv2: constants and five-role series selection

In [30]:
# DINOv2-specific settings. ViT inference has already finished, so PATCH_SIZE can now change from 16 to 14.
OUTPUT_ROOT = OUTPUT_DIR / "dino_inference"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
IMAGE_SIZE = 224
PATCH_SIZE = 14
BACKBONE_DIM = 384
EMBED_DIM = 256
FREEZE_EARLY_BLOCKS = int(bundle_manifest.get("freeze_early_blocks", 9))
N_ROLE_SLICES = 12
N_ROLE_WINDOWS = N_ROLE_SLICES - 2
FIVE_ROLE_ORDER = ["SAG_FLUID", "COR_FLUID", "AX_FLUID", "SAG_NONFLUID", "COR_NONFLUID"]
DINO_HUB_MODEL = "dinov2_vits14"
ENCODER_CHUNK_SIZE = 8
MIN_DECODED_SLICES = 4

IMAGE_SIZE = 224
PATCH_SIZE = 14
BACKBONE_DIM = 384
EMBED_DIM = 256
FREEZE_EARLY_BLOCKS = 9
N_ROLE_SLICES = 12
N_ROLE_WINDOWS = N_ROLE_SLICES - 2
FIVE_ROLE_ORDER = [
    "SAG_FLUID", "COR_FLUID", "AX_FLUID", "SAG_NONFLUID", "COR_NONFLUID"
]
DINO_HUB_MODEL = "dinov2_vits14"
ENCODER_CHUNK_SIZE = 8
MIN_DECODED_SLICES = 4

ROLE_SPECS = {
    "SAG_FLUID": ("sagittal", 1),
    "COR_FLUID": ("coronal", 1),
    "AX_FLUID": ("axial", 1),
    "SAG_NONFLUID": ("sagittal", 0),
    "COR_NONFLUID": ("coronal", 0),
}
PLANES = ["sagittal", "coronal", "axial"]

def normalize_plane(value):
    s = str(value).strip().lower()
    if "sag" in s:
        return "sagittal"
    if "cor" in s:
        return "coronal"
    if "axi" in s or "trans" in s or s == "ax":
        return "axial"
    return s

def parse_binary_flag(value):
    if pd.isna(value):
        return -1
    s = str(value).strip().lower()
    mapping = {
        "true": 1, "false": 0, "yes": 1, "no": 0,
        "y": 1, "n": 0, "1": 1, "0": 0,
        "1.0": 1, "0.0": 0,
    }
    if s in mapping:
        return mapping[s]
    try:
        x = float(s)
        return int(x) if x in (0.0, 1.0) else -1
    except Exception:
        return -1

test_series_df["plane_norm"] = test_series_df["Anatomical_Plane"].map(normalize_plane)
test_series_df["fluid_num"] = test_series_df["Fluid_Sensitive"].map(parse_binary_flag)
test_series_df["fat_num"] = test_series_df["Fat_Suppression"].map(parse_binary_flag)
test_series_df["SeriesInstanceUID"] = test_series_df["SeriesInstanceUID"].astype(str)

# Reproduce the two-stage selection used while building the training five-role index:
# 1) The original CNN-LSTM cache chose one primary series per plane by
#    Fluid_Sensitive DESC, Fat_Suppression DESC, SeriesInstanceUID ASC.
# 2) If that primary series did not match a role, Cell C selected a separate
#    fallback from the exact plane/fluid candidates with Fat_Suppression=0,
#    then unknown, then 1.
role_selection = {}
selection_rows = []
series_groups = {str(uid): group.copy() for uid, group in test_series_df.groupby(ID_COL, sort=False)}

for uid in test_df[ID_COL].astype(str).tolist():
    group = series_groups.get(uid, test_series_df.iloc[0:0])
    role_selection[uid] = {}
    primary_by_plane = {}

    for plane in PLANES:
        primary_candidates = group[group["plane_norm"].eq(plane)].copy()
        if primary_candidates.empty:
            primary_by_plane[plane] = None
        else:
            primary_candidates = primary_candidates.sort_values(
                ["fluid_num", "fat_num", "SeriesInstanceUID"],
                ascending=[False, False, True], kind="mergesort"
            )
            primary_by_plane[plane] = primary_candidates.iloc[0]

    for role in FIVE_ROLE_ORDER:
        plane, fluid_flag = ROLE_SPECS[role]
        candidates = group[
            group["plane_norm"].eq(plane) & group["fluid_num"].eq(fluid_flag)
        ].copy()
        candidate_count = len(candidates)
        primary = primary_by_plane.get(plane)
        selected_uid = None
        selection_source = "missing"

        if primary is not None and int(primary["fluid_num"]) == fluid_flag:
            selected_uid = str(primary["SeriesInstanceUID"])
            selection_source = "primary_cache_equivalent"
        elif not candidates.empty:
            candidates["_fat_rank"] = candidates["fat_num"].map({0: 0, -1: 1, 1: 2}).fillna(1)
            candidates = candidates.sort_values(
                ["_fat_rank", "SeriesInstanceUID"], ascending=[True, True], kind="mergesort"
            )
            selected_uid = str(candidates.iloc[0]["SeriesInstanceUID"])
            selection_source = "supplementary_fallback"

        role_selection[uid][role] = selected_uid
        selection_rows.append({
            ID_COL: uid,
            "role": role,
            "plane": plane,
            "fluid_flag": fluid_flag,
            "candidate_count": int(candidate_count),
            "primary_series_uid": str(primary["SeriesInstanceUID"]) if primary is not None else "",
            "SeriesInstanceUID": selected_uid or "",
            "selection_source": selection_source,
            "selection_status": "selected" if selected_uid else "no_candidate",
        })

role_selection_df = pd.DataFrame(selection_rows)
role_selection_df.to_csv(OUTPUT_ROOT / "test_role_selection.csv", index=False)
print("========== TEST ROLE SELECTION ==========")
display(role_selection_df.groupby(["role", "selection_source"]).size().rename("n_studies").reset_index())
print("Selected role series rows:", int(role_selection_df["SeriesInstanceUID"].ne("").sum()))

========== TEST ROLE SELECTION ==========


,role,selection_source,n_studies
0,AX_FLUID,primary_cache_equivalent,3
1,COR_FLUID,primary_cache_equivalent,3
2,COR_NONFLUID,missing,2
3,COR_NONFLUID,supplementary_fallback,1
4,SAG_FLUID,missing,1
5,SAG_FLUID,primary_cache_equivalent,2
6,SAG_NONFLUID,primary_cache_equivalent,1
7,SAG_NONFLUID,supplementary_fallback,2


Selected role series rows: 12


## 13. DINOv2: training-matched DICOM preprocessing

In [31]:
def safe_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return default

def natural_key(name):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", name)]

DICOM_HEADER_TAGS = [
    "ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber", "SOPInstanceUID",
    "PixelSpacing", "Rows", "Columns", "SeriesDescription", "ProtocolName", "SequenceName",
    "Laterality", "ImageLaterality", "FrameOfReferenceUID", "SliceThickness",
    "SpacingBetweenSlices", "PhotometricInterpretation", "RescaleSlope", "RescaleIntercept",
    "StudyInstanceUID", "SeriesInstanceUID",
]

def read_sorted_dicom_headers(series_dir: Path):
    files = sorted(p for p in series_dir.iterdir() if p.is_file() and p.suffix.lower() == ".dcm")
    if not files:
        files = sorted(p for p in series_dir.iterdir() if p.is_file() and not p.name.startswith("."))
    rows = []
    first_normal = None
    for path in files:
        try:
            ds = pydicom.dcmread(
                str(path), stop_before_pixels=True, force=True, specific_tags=DICOM_HEADER_TAGS
            )
            iop = getattr(ds, "ImageOrientationPatient", None)
            ipp = getattr(ds, "ImagePositionPatient", None)
            pos = np.nan
            instance = safe_float(getattr(ds, "InstanceNumber", np.nan))
            if iop is not None and ipp is not None and len(iop) >= 6 and len(ipp) >= 3:
                normal = np.cross(np.asarray(iop[:3], dtype=np.float64), np.asarray(iop[3:6], dtype=np.float64))
                norm = np.linalg.norm(normal)
                if norm > 0:
                    normal = normal / norm
                    if first_normal is None:
                        first_normal = normal
                    pos = float(np.dot(np.asarray(ipp[:3], dtype=np.float64), first_normal))
            rows.append({
                "path": path, "filename": path.name, "position_mm": pos,
                "instance": instance, "sop_uid": str(getattr(ds, "SOPInstanceUID", "")),
            })
        except Exception as exc:
            rows.append({
                "path": path, "filename": path.name, "position_mm": np.nan,
                "instance": np.nan, "sop_uid": "", "header_error": str(exc),
            })
    if not rows:
        return [], "no_dicom_files"
    n = len(rows)
    npos = sum(np.isfinite(r["position_mm"]) for r in rows)
    ninst = sum(np.isfinite(r["instance"]) for r in rows)
    if npos >= max(2, int(0.7 * n)):
        rows.sort(key=lambda r: (np.inf if not np.isfinite(r["position_mm"]) else r["position_mm"], r["filename"]))
        mode = "ImagePositionPatient_projected"
    elif ninst >= max(2, int(0.7 * n)):
        rows.sort(key=lambda r: (np.inf if not np.isfinite(r["instance"]) else r["instance"], r["filename"]))
        mode = "InstanceNumber"
    else:
        rows.sort(key=lambda r: natural_key(r["filename"]))
        mode = "filename_fallback"
    for i, row in enumerate(rows):
        row["sort_index"] = i
    return rows, mode

def decode_dicom_slice(path: Path):
    ds = pydicom.dcmread(str(path), force=True)
    pixels = np.asarray(ds.pixel_array)
    if pixels.ndim == 3:
        if pixels.shape[-1] in (3, 4):
            pixels = pixels[..., :3].astype(np.float32)
            pixels = 0.299 * pixels[..., 0] + 0.587 * pixels[..., 1] + 0.114 * pixels[..., 2]
        else:
            pixels = pixels[pixels.shape[0] // 2]
    pixels = np.squeeze(pixels)
    if pixels.ndim != 2:
        raise ValueError(f"Expected 2D MRI slice, got shape={pixels.shape}")
    pixels = pixels.astype(np.float32)
    try:
        pixels = np.asarray(apply_modality_lut(pixels, ds), dtype=np.float32)
    except Exception:
        slope = safe_float(getattr(ds, "RescaleSlope", 1.0), 1.0)
        intercept = safe_float(getattr(ds, "RescaleIntercept", 0.0), 0.0)
        pixels = pixels * slope + intercept
    if str(getattr(ds, "PhotometricInterpretation", "MONOCHROME2")).upper() == "MONOCHROME1":
        lo, hi = np.nanmin(pixels), np.nanmax(pixels)
        pixels = hi + lo - pixels
    pixels[~np.isfinite(pixels)] = 0
    return pixels, ds

def process_role_series(study_uid: str, series_uid: str):
    # Decode one selected series into 12 uint8 224x224 slices, matching training Cell D.
    zero_images = np.zeros((N_ROLE_SLICES, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8)
    zero_valid = np.zeros(N_ROLE_SLICES, dtype=np.uint8)
    zero_pos = np.zeros(N_ROLE_SLICES, dtype=np.float32)
    series_dir = TEST_SERIES_DIR / str(study_uid) / str(series_uid)
    if not series_dir.is_dir():
        return None, None, None, {"status": "missing_series_directory", "n_files": 0, "n_decoded": 0}
    headers, sort_mode = read_sorted_dicom_headers(series_dir)
    if not headers:
        return None, None, None, {"status": sort_mode, "n_files": 0, "n_decoded": 0}

    n = len(headers)
    initial = np.unique(np.linspace(0, n - 1, min(N_ROLE_SLICES, n)).round().astype(int)).tolist()
    initial_set = set(initial)
    candidates = initial + [i for i in range(n) if i not in initial_set]
    decoded_images, decoded_positions, errors = [], [], []
    for header_idx in candidates:
        header = headers[header_idx]
        try:
            image, ds = decode_dicom_slice(header["path"])
            actual_study = str(getattr(ds, "StudyInstanceUID", ""))
            actual_series = str(getattr(ds, "SeriesInstanceUID", ""))
            if actual_study and actual_study != str(study_uid):
                raise ValueError("StudyInstanceUID mismatch")
            if actual_series and actual_series != str(series_uid):
                raise ValueError("SeriesInstanceUID mismatch")
            decoded_images.append(image)
            decoded_positions.append(header["sort_index"] / max(n - 1, 1))
            if len(decoded_images) >= N_ROLE_SLICES:
                break
        except Exception as exc:
            errors.append(f"{header['filename']}:{type(exc).__name__}:{exc}")

    if len(decoded_images) < MIN_DECODED_SLICES:
        return None, None, None, {
            "status": "too_few_decoded_slices",
            "n_files": n,
            "n_decoded": len(decoded_images),
            "sort_mode": sort_mode,
            "error": " | ".join(errors[:2]),
        }

    order = np.argsort(decoded_positions, kind="stable")
    decoded_images = [decoded_images[i] for i in order]
    decoded_positions = [decoded_positions[i] for i in order]
    values = np.concatenate([image.reshape(-1) for image in decoded_images])
    lo, hi = np.percentile(values, [1.0, 99.5])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(values.min()), float(values.max())
    if hi <= lo:
        hi = lo + 1.0

    out_images = zero_images.copy()
    out_valid = zero_valid.copy()
    out_pos = zero_pos.copy()
    for slot, (image, position) in enumerate(zip(decoded_images, decoded_positions)):
        x = np.clip((np.clip(image, lo, hi) - lo) / (hi - lo), 0, 1)
        x = (x * 255.0).round().astype(np.uint8)
        out_images[slot] = cv2.resize(x, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
        out_valid[slot] = 1
        out_pos[slot] = float(position)
    return out_images, out_valid, out_pos, {
        "status": "ok", "n_files": n, "n_decoded": len(decoded_images),
        "sort_mode": sort_mode, "error": " | ".join(errors[:3]),
    }

print("DICOM preprocessing functions ready.")

DICOM preprocessing functions ready.


## 14. DINOv2: build five-role 2.5D windows

In [32]:
def build_study_windows(study_uid: str):
    # Returns X [5,10,3,224,224], valid [5,10], positions [5,10], and diagnostics.
    X = np.zeros(
        (len(FIVE_ROLE_ORDER), N_ROLE_WINDOWS, 3, IMAGE_SIZE, IMAGE_SIZE), dtype=np.uint8
    )
    window_valid = np.zeros((len(FIVE_ROLE_ORDER), N_ROLE_WINDOWS), dtype=np.uint8)
    window_positions = np.zeros((len(FIVE_ROLE_ORDER), N_ROLE_WINDOWS), dtype=np.float32)
    diagnostics = []

    for role_idx, role in enumerate(FIVE_ROLE_ORDER):
        series_uid = role_selection.get(str(study_uid), {}).get(role)
        if not series_uid:
            diagnostics.append({
                ID_COL: str(study_uid), "role": role, "SeriesInstanceUID": "",
                "status": "no_candidate_series", "n_files": 0, "n_decoded": 0,
            })
            continue
        slices, slice_valid, slice_pos, info = process_role_series(study_uid, series_uid)
        diagnostics.append({
            ID_COL: str(study_uid), "role": role, "SeriesInstanceUID": str(series_uid), **info
        })
        if slices is None:
            continue
        for window_idx in range(N_ROLE_WINDOWS):
            local_valid = slice_valid[window_idx:window_idx + 3]
            if int(local_valid.sum()) >= 2:
                X[role_idx, window_idx] = slices[window_idx:window_idx + 3]
                window_valid[role_idx, window_idx] = 1
                window_positions[role_idx, window_idx] = float(
                    slice_pos[window_idx:window_idx + 3][local_valid.astype(bool)].mean()
                )

    n_valid_windows = int(window_valid.sum())
    n_available_roles = int(window_valid.any(axis=1).sum())
    if n_valid_windows == 0:
        return None, None, None, diagnostics, n_available_roles
    return X, window_valid, window_positions, diagnostics, n_available_roles

print("One-study window builder ready.")
print("Expected model input: images [5,10,3,224,224], mask/positions [5,10].")

One-study window builder ready.
Expected model input: images [5,10,3,224,224], mask/positions [5,10].


## 15. DINOv2: exact five-role V2 architecture

In [33]:
# ============================================================
# CELL I — Five-role 2.5D DINOv2 model (V2: target x role pooling)
# Shared ViT-S/14 encoder; each target pools each role independently;
# five role vectors are concatenated and sent to a target-specific head.
# ============================================================
import math
import torch
import torch.nn as nn


class FiveRoleDINOv2_2p5D(nn.Module):
    def __init__(
        self,
        backbone,
        image_size=IMAGE_SIZE,
        backbone_dim=BACKBONE_DIM,
        embed_dim=EMBED_DIM,
        freeze_early_blocks=FREEZE_EARLY_BLOCKS,
        dropout=0.15,
        encoder_chunk_size=8,
    ):
        super().__init__()
        self.backbone = backbone
        self.image_size = int(image_size)
        self.backbone_dim = int(backbone_dim)
        self.embed_dim = int(embed_dim)
        self.encoder_chunk_size = int(encoder_chunk_size)
        self.n_roles = len(FIVE_ROLE_ORDER)
        self.n_targets = len(TARGETS)
        self.n_windows = N_ROLE_WINDOWS

        if self.image_size % PATCH_SIZE != 0:
            raise ValueError("Image size must be divisible by DINOv2 patch size 14.")
        if not hasattr(self.backbone, "blocks") or len(self.backbone.blocks) != 12:
            raise ValueError("Expected DINOv2 ViT-S/14 with 12 transformer blocks.")
        if self.n_roles != 5:
            raise ValueError(f"Expected exactly five roles, found {self.n_roles}.")

        # Preserve the existing fine-tuning policy: freeze the backbone first,
        # then unfreeze its last blocks and final LayerNorm.
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        if not 0 <= freeze_early_blocks < 12:
            raise ValueError("freeze_early_blocks must be between 0 and 11.")
        self.trainable_blocks = list(self.backbone.blocks[freeze_early_blocks:])
        for block in self.trainable_blocks:
            for p in block.parameters():
                p.requires_grad_(True)
        for p in self.backbone.norm.parameters():
            p.requires_grad_(True)

        # Convert each DINOv2 window into one compact feature vector.
        self.patch_attention = nn.Sequential(
            nn.LayerNorm(self.backbone_dim),
            nn.Linear(self.backbone_dim, 1, bias=False),
        )
        self.window_projection = nn.Sequential(
            nn.Linear(self.backbone_dim * 2, self.embed_dim),
            nn.LayerNorm(self.embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Mild spatial prior; peripheral patches remain available to attention.
        grid = self.image_size // PATCH_SIZE
        coords = torch.linspace(-1.0, 1.0, grid)
        yy, xx = torch.meshgrid(coords, coords, indexing="ij")
        log_gaussian = -0.5 * ((xx / 0.55) ** 2 + (yy / 0.55) ** 2)
        log_gaussian = log_gaussian - log_gaussian.max()
        self.register_buffer("center_bias", (0.15 * log_gaussian.reshape(-1)).float())

        self.role_embedding = nn.Parameter(
            torch.zeros(1, self.n_roles, 1, self.embed_dim)
        )
        self.window_embedding = nn.Embedding(self.n_windows, self.embed_dim)
        self.position_projection = nn.Sequential(
            nn.Linear(3, self.embed_dim),
            nn.GELU(),
            nn.Linear(self.embed_dim, self.embed_dim),
        )
        nn.init.normal_(self.role_embedding, std=0.02)

        # V2: independent attention query for every (target, role) pair.
        # Shape: [12 targets, 5 roles, embedding dimension].
        self.target_role_queries = nn.Parameter(
            torch.empty(self.n_targets, self.n_roles, self.embed_dim)
        )
        nn.init.normal_(self.target_role_queries, std=0.5)

        # h(target, role) = global_role + gamma * (attended_role - global_role).
        # Start gamma at 0.70, not 0.10, so target-specific evidence is not
        # almost suppressed at initialization.
        gamma_init = 0.70
        gamma_logit = math.log(gamma_init / (1.0 - gamma_init))
        self.target_role_residual_logits = nn.Parameter(
            torch.full((self.n_targets, self.n_roles), gamma_logit)
        )
        self.role_feature_norm = nn.LayerNorm(self.embed_dim)

        # Every target gets its own prediction head. Each head sees the five
        # role-specific vectors concatenated in FIVE_ROLE_ORDER, plus presence
        # indicators so a missing role is distinguishable from a real zero.
        head_input_dim = self.n_roles * self.embed_dim + self.n_roles
        self.target_classifiers = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(head_input_dim),
                nn.Linear(head_input_dim, 128),
                nn.GELU(),
                nn.Dropout(0.25),
                nn.Linear(128, 1),
            )
            for _ in range(self.n_targets)
        ])

        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        self.register_buffer("image_mean", mean)
        self.register_buffer("image_std", std)

    def train(self, mode=True):
        super().train(mode)
        # Frozen DINO blocks remain in eval mode; selected final blocks train.
        self.backbone.eval()
        for block in self.trainable_blocks:
            block.train(mode)
        self.backbone.norm.train(mode)
        return self

    def forward(self, images, window_valid, window_positions):
        # images: [B, 5, 10, 3, 224, 224], uint8
        # masks/positions: [B, 5, 10]
        if images.ndim != 6:
            raise ValueError(f"Expected 6D images, received {tuple(images.shape)}")
        B, R, W, C, H, WW = images.shape
        expected = (self.n_roles, self.n_windows, 3, self.image_size, self.image_size)
        if (R, W, C, H, WW) != expected:
            raise ValueError(
                f"Unexpected image shape {tuple(images.shape)}; expected [B, {expected}]."
            )
        if window_valid.shape != (B, R, W):
            raise ValueError("window_valid shape does not match images.")
        if window_positions.shape != (B, R, W):
            raise ValueError("window_positions shape does not match images.")
        if not window_valid.bool().any(dim=(1, 2)).all():
            raise ValueError("At least one study in the batch has no valid windows.")

        # Encode only valid windows; do not waste DINOv2 compute on padded slots.
        flat_valid = window_valid.bool().reshape(-1)
        valid_indices = flat_valid.nonzero(as_tuple=False).squeeze(1)
        flat_images = images.reshape(B * R * W, C, H, WW)
        selected = flat_images.index_select(0, valid_indices).float().div_(255.0)
        selected = (selected - self.image_mean) / self.image_std

        encoded_chunks = []
        for start in range(0, selected.shape[0], self.encoder_chunk_size):
            chunk = selected[start:start + self.encoder_chunk_size]
            feat = self.backbone.forward_features(chunk)
            cls_token = feat["x_norm_clstoken"]
            patch_tokens = feat["x_norm_patchtokens"]
            patch_scores = self.patch_attention(patch_tokens).squeeze(-1)
            patch_scores = patch_scores + self.center_bias.unsqueeze(0)
            patch_weights = torch.softmax(patch_scores, dim=1)
            patch_pool = torch.sum(
                patch_tokens * patch_weights.unsqueeze(-1), dim=1
            )
            encoded_chunks.append(
                self.window_projection(torch.cat([cls_token, patch_pool], dim=-1))
            )
        valid_features = torch.cat(encoded_chunks, dim=0)

        flat_features = valid_features.new_zeros((B * R * W, self.embed_dim))
        flat_features = flat_features.index_copy(0, valid_indices, valid_features)
        tokens = flat_features.view(B, R, W, self.embed_dim)

        # Add role, window-order and normalized slice-position information.
        pos = window_positions.to(dtype=tokens.dtype).clamp(0.0, 1.0)
        pos_features = torch.stack(
            [pos, torch.sin(2.0 * math.pi * pos), torch.cos(2.0 * math.pi * pos)],
            dim=-1,
        )
        pos_features = self.position_projection(pos_features)
        order_ids = torch.arange(W, device=images.device)
        tokens = (
            tokens
            + self.role_embedding.to(dtype=tokens.dtype)
            + self.window_embedding(order_ids).view(1, 1, W, self.embed_dim)
            + pos_features
        )

        # Per-role global means: [B, R, D]. Missing roles remain zero.
        valid = window_valid.bool()
        mask_f = valid.unsqueeze(-1).to(tokens.dtype)
        role_counts = mask_f.sum(dim=2).clamp_min(1.0)
        role_global = (tokens * mask_f).sum(dim=2) / role_counts

        # Independent attention over windows WITHIN each role for each target.
        # Scores: [B, T, R, W]; masked softmax safely handles absent roles.
        scores = torch.einsum(
            "brwd,trd->btrw", tokens, self.target_role_queries
        ) / math.sqrt(self.embed_dim)
        scores = scores.masked_fill(
            ~valid[:, None, :, :], torch.finfo(scores.dtype).min
        )
        attention = torch.softmax(scores.float(), dim=-1).to(tokens.dtype)
        attention = attention * valid[:, None, :, :].to(tokens.dtype)
        attention = attention / attention.sum(dim=-1, keepdim=True).clamp_min(1e-6)
        attended_role = torch.einsum("btrw,brwd->btrd", attention, tokens)

        gamma = torch.sigmoid(self.target_role_residual_logits).view(
            1, self.n_targets, self.n_roles, 1
        )
        target_role_features = role_global[:, None, :, :] + gamma * (
            attended_role - role_global[:, None, :, :]
        )
        target_role_features = self.role_feature_norm(target_role_features)

        # Keep missing-role vectors at zero after LayerNorm; the presence flag
        # below explicitly tells the target head that the role was unavailable.
        role_presence = valid.any(dim=-1).to(target_role_features.dtype)  # [B, R]
        target_role_features = target_role_features * role_presence[:, None, :, None]

        # Concatenate five independently pooled role vectors per target.
        target_role_features = target_role_features.reshape(
            B, self.n_targets, self.n_roles * self.embed_dim
        )
        presence_features = role_presence[:, None, :].expand(
            -1, self.n_targets, -1
        )
        classifier_inputs = torch.cat(
            [target_role_features, presence_features], dim=-1
        )

        # Independent nonlinear prediction head for each target.
        logits = torch.cat(
            [
                self.target_classifiers[t](classifier_inputs[:, t, :])
                for t in range(self.n_targets)
            ],
            dim=1,
        )
        return logits
print("FiveRoleDINOv2_2p5D class defined.")


FiveRoleDINOv2_2p5D class defined.


## 16. DINOv2: load five fold checkpoints (offline)

In [34]:
# Instantiate DINOv2 from the local source in the attached Dataset (or extracted ZIP); pretrained=False prevents network access.
try:
    backbone_template = torch.hub.load(
        str(DINOV2_REPO_DIR), DINO_HUB_MODEL, source="local", pretrained=False
    )
except Exception as exc:
    raise RuntimeError(
        "Could not instantiate DINOv2 from the bundled local repository. "
        "Internet is disabled; verify that the attached bundle contains the complete dinov2_repo/."
    ) from exc

if not hasattr(backbone_template, "forward_features") or len(backbone_template.blocks) != 12:
    raise RuntimeError("Bundled source did not create the expected DINOv2 ViT-S/14 architecture.")
backbone_template.eval()

models = []
fold_model_info = []
for fold, checkpoint_member in enumerate(DINO_FOLD_MEMBERS, start=1):
    # Checkpoints are read directly from the attached Dataset or streamed from the Input ZIP.
    with DINO_ZIP_HANDLE.open(checkpoint_member, "r") as checkpoint_stream:
        try:
            checkpoint = torch.load(checkpoint_stream, map_location="cpu", weights_only=False)
        except TypeError:
            checkpoint_stream.seek(0)
            checkpoint = torch.load(checkpoint_stream, map_location="cpu")

    if int(checkpoint.get("fold", -1)) != fold:
        raise ValueError(f"Checkpoint fold mismatch: expected {fold}, got {checkpoint.get('fold')}.")
    if list(checkpoint.get("targets", [])) != TARGETS:
        raise ValueError(f"Fold {fold}: target order mismatch.")
    if list(checkpoint.get("role_order", [])) != FIVE_ROLE_ORDER:
        raise ValueError(f"Fold {fold}: role order mismatch.")
    if "model_state" not in checkpoint:
        raise KeyError(f"Fold {fold}: model_state missing from checkpoint.")

    fold_backbone = copy.deepcopy(backbone_template)
    fold_model = FiveRoleDINOv2_2p5D(
        backbone=fold_backbone,
        image_size=IMAGE_SIZE,
        backbone_dim=BACKBONE_DIM,
        embed_dim=EMBED_DIM,
        freeze_early_blocks=FREEZE_EARLY_BLOCKS,
        encoder_chunk_size=ENCODER_CHUNK_SIZE,
    ).to(device)
    fold_model.load_state_dict(checkpoint["model_state"], strict=True)
    fold_model.eval()
    models.append(fold_model)
    fold_model_info.append({
        "fold": fold,
        "best_epoch": int(checkpoint.get("best_epoch", -1)),
        "best_val_loss": float(checkpoint.get("best_val_loss", np.nan)),
        "signature": str(checkpoint.get("signature", "")),
    })
    print(
        f"Loaded DINOv2 fold {fold}/5 | best epoch={fold_model_info[-1]['best_epoch']} "
        f"| best val BCE={fold_model_info[-1]['best_val_loss']:.5f}"
    )
    del checkpoint, fold_backbone, fold_model
    gc.collect()
    torch.cuda.empty_cache()

DINO_ZIP_HANDLE.close()
del backbone_template
gc.collect()
print("Loaded DINOv2 ensemble members:", len(models))
print("Parameters per fold (M):", round(sum(p.numel() for p in models[0].parameters()) / 1e6, 2))
print("Allocated GPU memory GiB:", round(torch.cuda.memory_allocated() / 1024**3, 2))


Loaded DINOv2 fold 1/5 | best epoch=2 | best val BCE=0.64359
Loaded DINOv2 fold 2/5 | best epoch=5 | best val BCE=0.57666
Loaded DINOv2 fold 3/5 | best epoch=5 | best val BCE=0.54640
Loaded DINOv2 fold 4/5 | best epoch=5 | best val BCE=0.65672
Loaded DINOv2 fold 5/5 | best epoch=9 | best val BCE=0.55387
Loaded DINOv2 ensemble members: 5
Parameters per fold (M): 24.35
Allocated GPU memory GiB: 0.47


## 17. DINOv2: inference on test studies

In [35]:
test_uids = test_df[ID_COL].astype(str).tolist()
N_TEST = len(test_uids)
fold_probabilities = np.full((N_TEST, len(models), len(TARGETS)), 0.5, dtype=np.float32)
role_diagnostics_all = []
study_inference_rows = []
start_time = time.time()

for study_idx, uid in enumerate(test_uids):
    X, valid_mask, positions, diagnostics, n_roles = build_study_windows(uid)
    role_diagnostics_all.extend(diagnostics)

    if X is None:
        # Neutral fallback only when no usable 2.5D window could be decoded.
        study_inference_rows.append({
            ID_COL: uid, "n_available_roles": int(n_roles),
            "status": "all_roles_unusable_fallback_0.5",
        })
    else:
        images_t = torch.from_numpy(X[None]).to(device, non_blocking=True)
        valid_t = torch.from_numpy(valid_mask[None]).to(device, non_blocking=True).bool()
        positions_t = torch.from_numpy(positions[None]).to(device, non_blocking=True)
        with torch.inference_mode():
            for model_idx, model in enumerate(models):
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                    logits = model(images_t, valid_t, positions_t)
                probs = torch.sigmoid(logits.float()).squeeze(0).cpu().numpy()
                fold_probabilities[study_idx, model_idx] = probs.astype(np.float32)
        study_inference_rows.append({
            ID_COL: uid, "n_available_roles": int(n_roles), "status": "ok"
        })
        del images_t, valid_t, positions_t

    if (study_idx + 1) % 10 == 0 or study_idx + 1 == N_TEST:
        elapsed_min = (time.time() - start_time) / 60.0
        print(
            f"Inference {study_idx + 1}/{N_TEST} studies | "
            f"elapsed={elapsed_min:.1f} min | usable roles={n_roles}",
            flush=True,
        )

ensemble_probabilities = fold_probabilities.mean(axis=1)
if ensemble_probabilities.shape != (N_TEST, len(TARGETS)):
    raise RuntimeError(f"Unexpected ensemble shape: {ensemble_probabilities.shape}")
if not np.isfinite(ensemble_probabilities).all():
    raise RuntimeError("Ensemble predictions contain NaN/Inf.")

role_diagnostics_df = pd.DataFrame(role_diagnostics_all)
study_inference_df = pd.DataFrame(study_inference_rows)
role_diagnostics_df.to_csv(OUTPUT_ROOT / "test_role_preprocessing_log.csv", index=False)
study_inference_df.to_csv(OUTPUT_ROOT / "test_study_inference_log.csv", index=False)
print("Inference finished in minutes:", round((time.time() - start_time) / 60.0, 2))
print("Studies without usable role windows:", int(study_inference_df["status"].ne("ok").sum()))
print("Mean available roles/study:", round(float(study_inference_df["n_available_roles"].mean()), 3))
print("Ensemble probability range:", float(ensemble_probabilities.min()), "to", float(ensemble_probabilities.max()))

# Keep the 5-fold mean and per-study validity for the final blend.
dino_probabilities = ensemble_probabilities.astype(np.float32)
dino_valid_mask = study_inference_df["status"].eq("ok").to_numpy(dtype=bool)
if dino_probabilities.shape != (N_TEST, len(TARGETS)):
    raise RuntimeError(f"DINOv2 prediction shape mismatch: {dino_probabilities.shape}")
print("DINOv2 inference output:", dino_probabilities.shape, "| valid studies:", int(dino_valid_mask.sum()), "/", N_TEST)


Inference 3/3 studies | elapsed=0.1 min | usable roles=5
Inference finished in minutes: 0.07
Studies without usable role windows: 0
Mean available roles/study: 4.0
Ensemble probability range: 0.17440757155418396 to 0.7892706394195557
DINOv2 inference output: (3, 12) | valid studies: 3 / 3


## 18. Blend model probabilities and create the competition submission

Default: 90% DINOv2 + 10% ViT-B/16. If only one model has a usable prediction for a study, that model is used alone; `0.5` is used only when both inference pipelines failed for that study.

In [36]:
if not (0.0 <= DINO_WEIGHT <= 1.0):
    raise ValueError("DINO_WEIGHT must be in [0, 1].")
if vit_probabilities.shape != dino_probabilities.shape:
    raise RuntimeError("ViT and DINOv2 predictions do not have matching shapes.")
if len(test_uids) != len(sample_submission):
    raise RuntimeError("Test UID count does not match sample_submission.")

both_ok = vit_valid_mask & dino_valid_mask
only_dino = dino_valid_mask & ~vit_valid_mask
only_vit = vit_valid_mask & ~dino_valid_mask
neither_ok = ~vit_valid_mask & ~dino_valid_mask


def make_blend(weight_dino: float) -> np.ndarray:
    out = np.full_like(dino_probabilities, 0.5, dtype=np.float32)
    out[both_ok] = (
        weight_dino * dino_probabilities[both_ok]
        + (1.0 - weight_dino) * vit_probabilities[both_ok]
    )
    out[only_dino] = dino_probabilities[only_dino]
    out[only_vit] = vit_probabilities[only_vit]
    return out


def make_submission(probabilities: np.ndarray) -> pd.DataFrame:
    if probabilities.shape != (N_TEST, len(TARGETS)):
        raise ValueError(f"Bad probability shape: {probabilities.shape}")
    if not np.isfinite(probabilities).all() or (probabilities < 0).any() or (probabilities > 1).any():
        raise ValueError("Blend contains invalid probability values.")
    pred_df = pd.DataFrame(probabilities, columns=TARGETS)
    pred_df.insert(0, ID_COL, test_uids)
    result = sample_submission[[ID_COL]].merge(pred_df, on=ID_COL, how="left", validate="one_to_one")
    result = result[EXPECTED_COLUMNS]
    if result[ID_COL].tolist() != sample_submission[ID_COL].tolist():
        raise RuntimeError("Final UID ordering differs from sample_submission.")
    if result.isna().any().any():
        raise RuntimeError("Final submission contains missing values.")
    return result

# Official submission artifact: exactly this file is generated by the notebook run.
final_probabilities = make_blend(DINO_WEIGHT)
submission = make_submission(final_probabilities)
SUBMISSION_PATH = WORK_ROOT / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False, float_format="%.8f")

# Store alternate blend candidates separately; Kaggle will use only submission.csv.
CANDIDATE_DIR = OUTPUT_DIR / "candidate_submissions"
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)
for w in CANDIDATE_DINO_WEIGHTS:
    candidate = make_submission(make_blend(float(w)))
    candidate_path = CANDIDATE_DIR / f"submission_dino_{int(round(w*100)):03d}_vit_{int(round((1-w)*100)):03d}.csv"
    candidate.to_csv(candidate_path, index=False, float_format="%.8f")
    print(f"Saved candidate DINO={w:.0%}, ViT={1-w:.0%}: {candidate_path}")

# Persist raw model probabilities for later comparison/debugging (not used as submission inputs).
pd.DataFrame({ID_COL: test_uids, **{f"dino__{t}": dino_probabilities[:,j] for j,t in enumerate(TARGETS)}, **{f"vit__{t}": vit_probabilities[:,j] for j,t in enumerate(TARGETS)}}).to_csv(OUTPUT_DIR/"model_probabilities.csv", index=False)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "competition_root": str(COMP_ROOT),
    "n_test_studies": int(N_TEST),
    "targets": TARGETS,
    "models": {
        "dino": {"bundle": str(dino_bundle_path), "architecture": bundle_manifest.get("architecture"), "n_folds": len(models), "valid_studies": int(dino_valid_mask.sum())},
        "vit": {"artifact": str(vit_artifact_path), "architecture": "ViT-B/16 Residual Attention Pooling", "valid_studies": int(vit_valid_mask.sum())},
    },
    "blend": {"dino_weight": float(DINO_WEIGHT), "vit_weight": float(1.0-DINO_WEIGHT), "fallback_policy": "use sole valid model; 0.5 only when both invalid"},
    "validity": {"both": int(both_ok.sum()), "dino_only": int(only_dino.sum()), "vit_only": int(only_vit.sum()), "neither": int(neither_ok.sum())},
    "submission_path": str(SUBMISSION_PATH),
}
(OUTPUT_DIR/"ensemble_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

print("========== ENSEMBLE SUBMISSION CREATED ==========")
print("Submission:", SUBMISSION_PATH)
print("Shape:", submission.shape)
print("DINO-only studies:", int(only_dino.sum()), "| ViT-only:", int(only_vit.sum()), "| both:", int(both_ok.sum()), "| neither:", int(neither_ok.sum()))
print("Probability min/max:", float(submission[TARGETS].min().min()), float(submission[TARGETS].max().max()))
display(submission.head())

Saved candidate DINO=100%, ViT=0%: /kaggle/working/model_ensemble_v2/candidate_submissions/submission_dino_100_vit_000.csv
Saved candidate DINO=90%, ViT=10%: /kaggle/working/model_ensemble_v2/candidate_submissions/submission_dino_090_vit_010.csv
Saved candidate DINO=80%, ViT=20%: /kaggle/working/model_ensemble_v2/candidate_submissions/submission_dino_080_vit_020.csv
Saved candidate DINO=70%, ViT=30%: /kaggle/working/model_ensemble_v2/candidate_submissions/submission_dino_070_vit_030.csv
========== ENSEMBLE SUBMISSION CREATED ==========
Submission: /kaggle/working/submission.csv
Shape: (3, 13)
DINO-only studies: 0 | ViT-only: 0 | both: 3 | neither: 0
Probability min/max: 0.1790870726108551 0.7498827576637268


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.179087,0.343645,0.324824,0.320004,0.490064,0.412520,0.323108,0.658871,0.499669,0.387476,0.317085,0.199742
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.415200,0.462887,0.423108,0.401426,0.620085,0.497222,0.535412,0.749883,0.686591,0.545932,0.417837,0.251993
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.190697,0.331941,0.443049,0.368950,0.543629,0.496980,0.573819,0.428080,0.338633,0.414279,0.341425,0.213929


## 19. Final Kaggle submission validation

In [37]:
check = pd.read_csv(SUBMISSION_PATH, dtype={ID_COL: str})
if check.shape != sample_submission.shape:
    raise RuntimeError(f"Saved submission shape mismatch: {check.shape} vs {sample_submission.shape}")
if check.columns.tolist() != EXPECTED_COLUMNS:
    raise RuntimeError("Saved submission columns/order do not match sample_submission.")
if check[ID_COL].tolist() != sample_submission[ID_COL].tolist():
    raise RuntimeError("Saved submission UID order does not match sample_submission.")
values = check[TARGETS].to_numpy(dtype=np.float64)
if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
    raise RuntimeError("Saved submission contains invalid scores.")
print("KAGGLE SUBMISSION OUTPUT: PASS")
print("File:", SUBMISSION_PATH)
print("Rows:", len(check), "| columns:", len(check.columns))
print("Bytes:", SUBMISSION_PATH.stat().st_size)
print("Internet: OFF | Training: not performed")
display(check.head())

KAGGLE SUBMISSION OUTPUT: PASS
File: /kaggle/working/submission.csv
Rows: 3 | columns: 13
Bytes: 722
Internet: OFF | Training: not performed


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.179087,0.343645,0.324824,0.320004,0.490064,0.412520,0.323108,0.658871,0.499669,0.387476,0.317085,0.199742
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.415200,0.462887,0.423108,0.401426,0.620085,0.497222,0.535412,0.749883,0.686591,0.545932,0.417837,0.251993
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.190697,0.331941,0.443049,0.368950,0.543629,0.496980,0.573819,0.428080,0.338633,0.414279,0.341425,0.213929
